<a href="https://colab.research.google.com/github/rahavi-r31/ExporterAI_Chapter_68_analytics/blob/colab/cleaning_file_prototype_feb_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# STEP 0: Imports & Global Configuration
# Purpose:
# - Centralize all imports
# - Define global configs used across the pipeline
# - Fail early with meaningful errors if environment is broken
# ============================================================

# ----------------------------
# 0.1 Standard library imports
# ----------------------------
import os
import sys
import warnings
from typing import List, Dict, Tuple
import re
import numpy as np
import pandas as pd

# Silence non-critical warnings for presentation clarity
warnings.filterwarnings("ignore")

# ----------------------------
# 0.2 Third-party imports
# ----------------------------
try:
    import pandas as pd
except ImportError as e:
    raise ImportError(
        "pandas is not installed. In Colab, run: !pip install pandas"
    ) from e

try:
    import numpy as np
except ImportError as e:
    raise ImportError(
        "numpy is not installed. In Colab, run: !pip install numpy"
    ) from e

# Colab-specific (safe import)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# ----------------------------
# 0.3 Pandas display & behavior config
# ----------------------------
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 50)

# Prevent chained assignment issues during cleaning
pd.options.mode.chained_assignment = None


# ----------------------------
# 0.4 Global pipeline configuration
# ----------------------------

# HS code we are focusing on (string to avoid numeric issues)
TARGET_HS_PREFIX = "6815"

# Accepted file formats for upload
ALLOWED_FILE_EXTENSIONS = (".xlsx", ".xls", ".csv")

# Minimum rows expected after HS filtering
# (used later for sanity checks)
MIN_EXPECTED_ROWS = 10

# Error values to replace with NaN
ERROR_VALUES = [
    "#REF!", "#NAME?", "#N/A", "#VALUE!", "#DIV/0!", "#NULL!",
    "nana", "NA", "N/A", "Na", "N A", "N?A", "n/a",
    "Null", "NULL", "null", "UNKNOWN", "Unknown", "unknown",
    "NOT FOUND", "Not Found", "not found",
    "0000", "00000000", "000000",
    "NIL", "Nil", "nil", "NONE", "None", "none",
    "-", "--", "---", "____", "X", "XX", "XXX"
]

# Unit standardization mapping
UNIT_CORRECTIONS = {
    'KGA': 'KGS', 'KILO': 'KGS', 'KG': 'KGS', 'KILOGRAMS': 'KGS',
    'MET': 'MTR', 'METER': 'MTR', 'METRES': 'MTR', 'METERS': 'MTR',
    'TONNE': 'MTS', 'TON': 'MTS', 'MT': 'MTS', 'TONS': 'MTS',
    'PIECE': 'PCS', 'PC': 'PCS', 'PIECES': 'PCS',
    'NUMBER': 'NOS', 'NO': 'NOS', 'NUM': 'NOS', 'NUMBERS': 'NOS',
    'DOZEN': 'DOZ', 'DZ': 'DOZ', 'PAIR': 'PRS', 'PAIRS': 'PRS',
    'SETS': 'SET', 'SQM': 'SQM', 'SQMTR': 'SQM', 'SQF': 'SQF', 'SQFT': 'SQF',
    'GROSS': 'GRS', 'FEET': 'FTS', 'FT': 'FTS',
    'CENTIMETER': 'CMS', 'CM': 'CMS', 'KILOMETER': 'KME', 'KM': 'KME',
    'CARTON': 'CTN', 'CTNS': 'CTN', 'LITER': 'LTR', 'LITRE': 'LTR', 'LTS': 'LTR'
}

# Country normalization mapping (expanded from chat)
COUNTRY_MAPPING = {
    "UNITED STATES": ["UNITED STATES", "UNITED STATES OF AMERICA", "USA", "U.S.A", "AMERICA"],
    "UNITED KINGDOM": ["UNITED KINGDOM", "UK", "GREAT BRITAIN"],
    "UNITED ARAB EMIRATES": ["UNITED ARAB EMIRATES", "UAE", "UNITED ARAB E"],
    "SAUDI ARABIA": ["SAUDI ARABIA", "KSA"],
    "IRAN": ["IRAN", "IRAN ISLAMIC REP"],
    "IRAQ": ["IRAQ"],
    "QATAR": ["QATAR"],
    "KUWAIT": ["KUWAIT"],
    "OMAN": ["OMAN"],
    "BAHRAIN": ["BAHRAIN"],
    "YEMEN": ["YEMEN", "YEMEN DEMOCRATIC"],
    "JORDAN": ["JORDAN"],
    "ISRAEL": ["ISRAEL"],
    "LEBANON": ["LEBANON"],
    "TURKEY": ["TURKEY", "TURKIYE"],
    "EGYPT": ["EGYPT"],
    "CHINA": ["CHINA"],
    "HONG KONG": ["HONG KONG", "HONGKONG"],
    "INDIA": ["INDIA"],
    "SINGAPORE": ["SINGAPORE"],
    "MALAYSIA": ["MALAYSIA"],
    "INDONESIA": ["INDONESIA"],
    "THAILAND": ["THAILAND"],
    "VIETNAM": ["VIETNAM", "VIETNAM DEMOCRATIC REP"],
    "GERMANY": ["GERMANY"],
    "FRANCE": ["FRANCE"],
    "ITALY": ["ITALY"],
    "SPAIN": ["SPAIN"],
    "NETHERLANDS": ["NETHERLANDS"],
    "BELGIUM": ["BELGIUM"],
    "AUSTRIA": ["AUSTRIA"],
    "SWITZERLAND": ["SWITZERLAND"],
    "SWEDEN": ["SWEDEN"],
    "NORWAY": ["NORWAY"],
    "DENMARK": ["DENMARK"],
    "POLAND": ["POLAND"],
    "RUSSIA": ["RUSSIA", "RUSSIAN FEDERATION"],
    "JAPAN": ["JAPAN"],
    "SOUTH KOREA": ["SOUTH KOREA", "KOREA REPUBLIC OF", "KOREA REPUBLIC OF"],
    "BRAZIL": ["BRAZIL"],
    "MEXICO": ["MEXICO"],
    "CANADA": ["CANADA"],
    "AUSTRALIA": ["AUSTRALIA"],
    "NEW ZEALAND": ["NEW ZEALAND"],
}

# Gulf & Middle East countries (explicit list, not fuzzy)
GULF_COUNTRIES = {
    "UNITED ARAB EMIRATES",
    "SAUDI ARABIA",
    "QATAR",
    "KUWAIT",
    "OMAN",
    "BAHRAIN",
}

MIDDLE_EAST_COUNTRIES = {
    "IRAN",
    "IRAQ",
    "JORDAN",
    "ISRAEL",
    "TURKEY",
    "EGYPT",
    "LEBANON",
    "YEMEN",
}

GULF_MIDDLE_EAST = GULF_COUNTRIES | MIDDLE_EAST_COUNTRIES


# ----------------------------
# 0.5 Canonical column definitions
# (Used later for selection & mapping)
# ----------------------------

COLUMN_SELECTION_MAP: Dict[str, List[str]] = {
    # Product Information
    "PRODUCT_DESCRIPTION": [
        "PRODUCT DESCRIPTION", "ITEM", "ITEM DESCRIPTION",
        "ITEM DESCRIPTIONS", "PRODUCT_DESCRIPTION", "DESCRIPTION",
        "GOODS DESCRIPTION", "COMMODITY", "PRODUCT", "COMMODITY DESCRIPTION", "PRODUCTDESCRIPITION",
        "RITC DESCRIPTION", "PRODUCTDESCRIPITION","GOODSDESCRIPTION", "HSN_DESCRIPTION", 'ITEM_DESCRIPTION',
        'ITEM_CATEGORY_DESCRIPTION', "ITME", "PRODUCT_DESCRIPITION"
    ],
    "HS_CODE": [
        "HS CODE", "HS_CODE", "HSN_CODE", "RITC", "HSCODE",
        "HSN CODE", "HSN", "TARIFF CODE", "HS", "RITCCODE", "RITC_8", "RITC_CODE"
    ],
    "CHAPTER": [
        "CH", "CHAPTER", "2 DIGIT", "HSCODE(2 DIGIT)", "HS2",
        "CHAPTER CODE", "2DIGIT", "CHAPTER'S","CHEPTER"
    ],

    # Quantity & Value
    "QUANTITY": [
        "QUANTITY", "PRODUCT_QUANTITY", "QTY", "QTY."
    ],
    "UNIT": [
        "UQC", "UNIT", "QUANTITY_UNIT", "UNIT QUANTITY", "UOM",
        "UNIT OF MEASURE", "QUANTITY UNIT", "UNITOFMEASUREMENT", "UNITQUANTITY","UNIT_QUANTITY"
    ],
    "UNIT_RATE_FC": [
        "ITEM_RATE", "UNIT RATE IN FC", "UNT PRICE FC", "UNIT PRICE (USD)",
        "UNIT RATE", "UNIT PRICE", "RATE", "UNIT_VALUE_USD", "UNIT_VALUE_FC", "UNIT_RATE_USD",
        "UNITPRICE", "UNIT PRICE FOREIGN", "UNIT PRICE FC", "ITEM_RATE_IN_FC","UNIT RATE IN FOREIGN CURRENCY",

    ],
    "UNIT_RATE_INR": [
        "UNIT RATE IN INR", "UNIT PRICE IN INR", "RATE INR","PER UNIT FOB", "UNT PRICE INR",
        "ITEM_RATE", "UNIT_VALUE_INR", "UNIT_VALUE_IN_INR", "UNIT_RATE_IN_INR"
    ],
    "FOB_INR": [
        "FOB", "FOB INR", "FOB IN INR", "FOB VALUE (INR)", "FOB_IN_INR",
        "FOB VALUE", "FOB_VALUE", "VALUE INR", "INR VALUE", "FOB IN INR.1", "FOBVALUEINRS",
        "TOTAL_VALUE_IN_INR", "TOTAL FOB VALUE IN INR"
    ],
    "FOB_FC": [
        "FOB IN FC", "INV VALUE FC", "FOB FC", "VALUE IN FC",
        "FOB USD", "USD VALUE", "FC VALUE", "TOTAL VALUE IN FC",
        "TOTAL_VALUE_IN_FC", "TOTAL_VALUE_USD", "TOTAL_VALUE_FC","TOTAL_VALUE_IN_USD"
    ],
    "CURRENCY": [
        "CURRENCY", "CURR", "CUR", "CURRENCY CODE", "CURR", "CURRENCY_NAME","UNIT RATE CURRENCY"
    ],

    # Exporter Information
    "EXPORTER_NAME": [
        "EXPORTER", "EXPORTER NAME", "EXPORTER NAMES", "EXPORTER_NAME",
        "SHIPPER", "SHIPPER NAME", "SUPPLIER", "SELLER", "EXPORTER_PERSON_NAME", "EXPORTERNAME",
        "INDIAN EXPORTER NAME","EXPORTER_NAME"
    ],
    "EXPORTER_ID": [
        "EXPORTER ID", "IEC", "IEC CODE", "IEC NO", "EXPORTER_ID",
        "IE CODE", "IMPORTER EXPORTER CODE", "IECNO", "IEC_NO"
    ],
    "EXPORTER_ADDRESS": [
        "EXPORTER ADDRESS", "Exporter_Address", "EXPORTER ADD",
        "EXPORTER ADD1", "Exporter Add1", "SHIPPER ADDRESS", "SHIPPER'S ADDRESS", "EXPORTER_ADDRESS.1",
        "ADDRESS", "EXPORTER ADDRESS & STATE"
    ],
    "EXPORTER_CITY_STATE": [
        "EXPORTER CITY/ STATE", "EXPORTER CITY", "Exporter_City_State",
        "EXPORTER STATE", "Exporter City", "EXPORTER CITY/STATE", "CITY STATE", 'EXPORTER_CITY_STATE.1',
        'EXPORTER_STATE','CITY/ STATE',"EXPORTER_CITY_STATE","CITY/ STATE","EXPORTER_CITY_STATE",
        "EXPORTER ADD2"
    ],
    "EXPORTER_PINCODE": [
        "EXPORTER PIN", "Exporter_PIN", "EXPORTER PINCODE", "PIN CODE", "PIN", "EXPORTER_PIN",
        "PIN_CODE", 'EXPORTER PIN CODE','EXPORTER_PINCODE'
    ],
    "EXPORTER_CONTACT_PERSON": [
        "CONTACT PERSON", "CONTACT PERSON2", "Exporter_Person_Name", "CONTACT PERSON NAME",
    ],
    "EXPORTER_CONTACT_EMAIL": [
        "EMAIL ID", "Exporter_Email", "EMAIL", "E-MAIL", "EXPORTER EMAIL", "EXPORTER_EMAIL", "EMAILID",
        "EMAIL"
    ],
    "EXPORTER_CONTACT_PHONE": [
        "CONTACT NO.", "Exporter_Contact", "PHONE", "MOBILE", "CONTACT", "EXPORTER PHONE", "EXPORTER_PHONE",
        "PHONE", "EXPORTER_CONTACT", 'CONTACTNO'
    ],

    # Importer Information
    "IMPORTER_NAME": [
        "IMPORTER NAME", "IMPORTER NAMES", "IMPORTER_NAME", "CONSIGNEE",
        "IMPORTER", "BUYER", "BUYER NAME", "IMPORTER NAME ", "CONSIGNEE NAME",'CONSINEENAME','CONSIGNEENAME',
        "CONSINEE_NAME", "CONSIGNEE_NAME","FOREIGN IMPORTER NAME","CONSINEE_NAME","FOREIGN IMPORTER NAME ADDRESS"
    ],
    "IMPORTER_ADDRESS": [
        "IMPORTER ADDRESS", "Consignee_Address", "BUYER ADDRESS", "CONSIGNEE_ADDRESS",
        "CONSINEEADDRESS","CONSIGNEE_ADDRESS4", "ADDRESS2", "CONSIGNEE ADD","CONSIGNEEADDRESS",
        "CONSINEE_ADDRESS","FOR_ADD1",

    ],

    # Port & Location
    "INDIAN_PORT": [
        "PORT CODE", "ORIGIN PORT", "INDIAN PORT",
        "ORIGIN_PORT_CODE", "CUSH", "PORT", "LOADING PORT", "PORT OF LOADING",
        "LOCATION1", "LOCATION", "SOURCE_PORT", 'INDIAN_PORT', "INDIAN PORT NAME","PORT OF ORIGIN"
    ],
    "FOREIGN_PORT": [
        "FOREIGN PORT", "DESTINATION PORT", "DISCHARGE PORT",
        "POD", "PORT_CD", "PORT OF DISCHARGE",
        "DESTINATION_PORT", "FORIGN PORT",
        "FOREIGNPORT", "PORT OF DESTINATION","FOREIGN_PORT"
    ]
    ,
    "COUNTRY": [
        "COUNTRY", "FOREIGN COUNTRY", "DESTINATION_COUNTRY",
        "DEST COUNTRY", "DESTINATION", "COUNTRY OF DESTINATION",
        "COUNTRY OF ORIGIN", "CONSIGNEE COUNTRY", "SOURCE_COUNTRY", "ORIGIN_COUNTRY",
        "FOREIGNCOUNTRY", "COUNTRYOFDESTINATIONNAME", "FOREIGN_COUNTRY", "CTRY OF DESTINATION"
    ],
    "MODE_OF_PORT": [
        "MODE OF PORT", "SHIPMENT MODE", "MODE", "TRANSPORT MODE", "MODE_OF_TRANSPORT"
    ],

    # Date & Time
    "SB_DATE": [
        "SBDT", "SB DATE", "SBDATE", "SHIPPING_DATE", "SHIPPING BILL DATE",
        "SHIPPING DATE", "DATE", "EXPORT DATE", "SB_DATE", "DATE", "SB_DT"
    ],
    "MONTH": ["MONTH", "MON", "MM"],
    "YEAR": ["YEAR", "YR", "YYYY"],

    # Document References
    "SB_NO": [
        "SBNO", "SB NO", "SHIPPING BILL NO", "SHIPPING_BILL_NO",
        "SB NUMBER", "BILL NO", "SBNUMBER", "SYSTEM_ID", "SB.NO."
    ],
    "INVOICE_NO": [
        "INVOICE_NO", "INVOICE NO", "INV NO", "INVOICE NUMBER", "INVOICE NO.",'INVOICE_NUMBER'
    ],
    "ITEM_NO": ["ITEM_NO", "ITEM NO", "LINE NO", "SR NO", "ITEM NUMBER"],

    # Other
    "DRAWBACK": ["DRAWBACK", "DWARBACK", "DBK", "DRAWBAKDVALUE", "DWARBACK"],
    "TYPE": ["TYPE"],
    "UID": ["UID"],
    "ID": ["ID"],
}



# ----------------------------
# 0.6 Utility: safe logging helper
# ----------------------------
def log(message: str) -> None:
    """
    Lightweight logger for Colab execution.
    Keeps output readable for presentations.
    """
    print(f"[INFO] {message}")


log("STEP 0 completed: Imports and configuration loaded successfully.")

# Environment summary (useful for demo confidence)
log(f"Running in Colab: {IN_COLAB}")
log(f"Target HS prefix: {TARGET_HS_PREFIX}")
log(f"Allowed file types: {ALLOWED_FILE_EXTENSIONS}")


[INFO] STEP 0 completed: Imports and configuration loaded successfully.
[INFO] Running in Colab: True
[INFO] Target HS prefix: 6815
[INFO] Allowed file types: ('.xlsx', '.xls', '.csv')


In [ ]:
# ============================================================
# STEP 1: Upload monthly files using Colab UI
# Purpose:
# - Allow user to upload multiple monthly files at once
# - Accept XLS, XLSX, CSV formats
# - Validate uploads early and fail fast
# - Return uploaded objects + validated filenames
# ============================================================

from typing import Tuple, Dict, List


def upload_files() -> Tuple[Dict[str, bytes], List[str]]:
    """
    Handles file upload via Google Colab UI.

    Returns
    -------
    uploaded : Dict[str, bytes]
        Dictionary of uploaded files where:
        - key   = filename
        - value = file content in bytes

    valid_files : List[str]
        List of filenames that match allowed extensions
        (XLS, XLSX, CSV)

    Raises
    ------
    EnvironmentError
        If not running inside Google Colab.
    ValueError
        If no files are uploaded or no valid files are found.
    """

    # ----------------------------
    # 1.0 Environment guard
    # ----------------------------
    if not IN_COLAB:
        raise EnvironmentError(
            "File upload via UI is supported only in Google Colab."
        )

    log("Please upload monthly import files (XLS / XLSX / CSV).")

    # ----------------------------
    # 1.1 Trigger Colab upload UI
    # ----------------------------
    uploaded: Dict[str, bytes] = files.upload()

    # ----------------------------
    # 1.2 Validate upload presence
    # ----------------------------
    if not uploaded:
        raise ValueError(
            "No files uploaded. Please upload at least one monthly data file."
        )

    log(f"Total files uploaded: {len(uploaded)}")

    # ----------------------------
    # 1.3 Filter allowed file types
    # ----------------------------
    valid_files: List[str] = []

    for filename in uploaded.keys():
        if filename.lower().endswith(ALLOWED_FILE_EXTENSIONS):
            valid_files.append(filename)
        else:
            log(f"Skipped unsupported file type: {filename}")

    if not valid_files:
        raise ValueError(
            "No valid data files found. Allowed formats: XLS, XLSX, CSV."
        )

    log(f"Valid data files detected: {len(valid_files)}")

    # ----------------------------
    # 1.4 Display uploaded file summary (sanity check)
    # ----------------------------
    print("\nUploaded Files Summary:")
    print("-" * 40)

    for fname in valid_files:
        size_kb = round(len(uploaded[fname]) / 1024, 2)
        print(f"{fname}  |  {size_kb} KB")

    print("-" * 40)

    # ----------------------------
    # 1.5 Return clean outputs
    # ----------------------------
    return uploaded, valid_files


In [ ]:
# ============================================================
# STEP 2: Read each uploaded file, map columns, clean HS code,
#         and append selected columns into a single DataFrame
# Purpose:
# - Read each validated uploaded file independently
# - Standardize column names
# - Map raw columns to canonical schema
# - Clean HS_CODE safely as string
# - Filter TARGET_HS_PREFIX early (performance + correctness)
# - Append all valid data into a single raw_df
# ============================================================

from typing import Dict, List


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Uppercase and strip column names to make mapping robust.
    """
    df = df.copy()
    df.columns = (
        df.columns
        .astype(str)
        .str.upper()
        .str.strip()
    )
    return df


def map_columns(df: pd.DataFrame, mapping: Dict[str, List[str]]) -> pd.DataFrame:
    """
    Create a new DataFrame with canonical columns based on COLUMN_SELECTION_MAP.
    Missing columns are created as NaN.
    """
    mapped_df = pd.DataFrame(index=df.index)

    for canonical_col, variants in mapping.items():
        found = False
        for v in variants:
            if v in df.columns:
                mapped_df[canonical_col] = df[v]
                found = True
                break

        if not found:
            mapped_df[canonical_col] = np.nan

    return mapped_df


def ingest_all_files(
    uploaded: Dict[str, bytes],
    valid_files: List[str]
) -> pd.DataFrame:
    """
    Ingest, map, HS-filter, and append all uploaded files.

    Parameters
    ----------
    uploaded : Dict[str, bytes]
        Files uploaded via Colab UI
    valid_files : List[str]
        Validated filenames (xls, xlsx, csv)

    Returns
    -------
    raw_df : pd.DataFrame
        Appended raw dataframe after column mapping
        and early HS filtering

    Raises
    ------
    ValueError
        If no valid HS data is found across all files
    """

    log("Starting STEP 2: File-wise ingestion with column mapping and HS filtering")

    processed_dfs: List[pd.DataFrame] = []

    # ----------------------------
    # 2.1 Process each file independently
    # ----------------------------
    for fname in valid_files:
        log(f"Processing file: {fname}")

        # ---- Safe file read
        try:
            if fname.lower().endswith((".xlsx", ".xls")):
                df = pd.read_excel(fname)
            elif fname.lower().endswith(".csv"):
                df = pd.read_csv(fname)
            else:
                log(f"Unsupported file skipped: {fname}")
                continue
        except Exception as e:
            log(f"Failed to read file {fname}: {e}")
            continue

        if df.empty:
            log(f"Empty file skipped: {fname}")
            continue

        # ---- Standardize column names
        df = standardize_columns(df)

        # ---- Map columns to canonical schema
        mapped_df = map_columns(df, COLUMN_SELECTION_MAP)

        # ---- Attach source file info (traceability)
        mapped_df["SOURCE_FILE"] = fname

        # ---- Clean HS_CODE (string safety only, no truncation yet)
        mapped_df["HS_CODE"] = (
            mapped_df["HS_CODE"]
            .astype(str)
            .str.replace(r"\.0$", "", regex=True)
            .str.strip()
        )

        # ---- Early HS filtering
        before_rows = len(mapped_df)
        mapped_df = mapped_df[
            mapped_df["HS_CODE"].str.startswith(TARGET_HS_PREFIX, na=False)
        ]
        after_rows = len(mapped_df)

        log(f"Rows before HS filter: {before_rows} | after: {after_rows}")

        # ---- Skip file if no relevant rows
        if mapped_df.empty:
            log(f"No HS {TARGET_HS_PREFIX} data found in {fname}")
            continue

        processed_dfs.append(mapped_df)

    # ----------------------------
    # 2.2 Append all processed files
    # ----------------------------
    if not processed_dfs:
        raise ValueError(
            f"No data found for HS code starting with {TARGET_HS_PREFIX} "
            "across all uploaded files."
        )

    raw_df = pd.concat(processed_dfs, ignore_index=True)

    log(f"STEP 2 completed: Total appended rows = {len(raw_df)}")

    # Structural sanity check
    print("\nFinal columns after STEP 2:")
    print(raw_df.columns.tolist())

    return raw_df


In [ ]:
# ============================================================
# STEP 3: Raw data statistics (baseline sanity metrics)
# ============================================================

def step_3_raw_data_statistics(raw_df: pd.DataFrame) -> Tuple[Dict, pd.Series]:
    """
    Compute baseline statistics for the raw EXIM dataset.

    Parameters:
    ----------
    raw_df : pd.DataFrame
        The raw appended DataFrame after STEP 2 ingestion.

    Returns:
    -------
    raw_stats : dict
        Dictionary containing total rows, missing percentages, FOB stats,
        distinct importers, countries, and units.
    unit_distribution_raw : pd.Series
        Top 15 units by count.
    """
    log("Starting STEP 3: Capturing raw data statistics")

    raw_stats = {}

    # ----------------------------
    # Total rows
    # ----------------------------
    raw_stats["total_rows"] = len(raw_df)

    # ----------------------------
    # Critical columns to monitor
    # ----------------------------
    CRITICAL_COLUMNS = [
        "HS_CODE",
        "IMPORTER_NAME",
        "COUNTRY",
        "FOB_INR",
        "QUANTITY",
        "UNIT"
    ]

    # Ensure only existing columns are used (prevents KeyError)
    existing_critical_cols = [c for c in CRITICAL_COLUMNS if c in raw_df.columns]
    missing_critical_cols = list(set(CRITICAL_COLUMNS) - set(existing_critical_cols))

    if missing_critical_cols:
        log(f"⚠️ Missing critical columns detected: {missing_critical_cols}")

    # ----------------------------
    # Missing percentage per critical column
    # ----------------------------
    missing_stats = (
        raw_df[existing_critical_cols]
        .isna()
        .mean()
        .mul(100)
        .round(2)
        .to_dict()
    )

    # Explicitly mark missing columns
    for col in missing_critical_cols:
        missing_stats[col] = "COLUMN_NOT_FOUND"

    raw_stats["missing_percentage"] = missing_stats

    # ----------------------------
    # FOB stats (numeric-safe)
    # ----------------------------
    if "FOB_INR" in raw_df.columns:
        fob_numeric = pd.to_numeric(raw_df["FOB_INR"], errors="coerce")
        raw_stats["avg_fob"] = fob_numeric.mean()
        raw_stats["median_fob"] = fob_numeric.median()
    else:
        raw_stats["avg_fob"] = None
        raw_stats["median_fob"] = None
        log("⚠️ FOB_INR column not found — FOB stats skipped")

    # ----------------------------
    # Cardinality checks
    # ----------------------------
    raw_stats["distinct_importers"] = (
        raw_df["IMPORTER_NAME"].nunique(dropna=True)
        if "IMPORTER_NAME" in raw_df.columns else None
    )

    raw_stats["distinct_countries"] = (
        raw_df["COUNTRY"].nunique(dropna=True)
        if "COUNTRY" in raw_df.columns else None
    )

    raw_stats["distinct_units"] = (
        raw_df["UNIT"].nunique(dropna=True)
        if "UNIT" in raw_df.columns else None
    )

    # ----------------------------
    # Unit diversity snapshot
    # ----------------------------
    if "UNIT" in raw_df.columns:
        unit_distribution_raw = (
            raw_df["UNIT"]
            .astype(str)
            .str.strip()
            .value_counts(dropna=True)
            .head(15)
        )
    else:
        unit_distribution_raw = pd.Series(dtype="int")
        log("⚠️ UNIT column not found — unit distribution skipped")

    # ----------------------------
    # Logging output
    # ----------------------------
    log("Raw data snapshot:")
    for k, v in raw_stats.items():
        print(f"{k}: {v}")

    print("\nTop raw units:")
    print(unit_distribution_raw)

    return raw_stats, unit_distribution_raw


In [ ]:
# ============================================================
# STEP 4.1: Global error value replacement
# ============================================================

def step_4_1_global_error_replacement(raw_df):
    """
    Replaces known error / placeholder values across the dataset with NaN.
    This runs as a global sanitation step before column-level cleaning.
    """

    log("Starting STEP 4.1: Replacing known error values with NaN")

    ERROR_VALUES = [
        "#REF!", "#NAME?", "#N/A", "#VALUE!", "#DIV/0!", "#NULL!",
        "nana", "NA", "N/A", "Na", "N A", "N?A", "n/a",
        "Null", "NULL", "null", "UNKNOWN", "Unknown", "unknown",
        "NOT FOUND", "Not Found", "not found",
        "0000", "00000000", "000000",
        "NIL", "Nil", "nil", "NONE", "None", "none",
        "-", "--", "---", "____", "X", "XX", "XXX"
    ]

    # Defensive copy safety (optional but prevents side-effects)
    df = raw_df.copy()

    # Replace exact matches
    df.replace(ERROR_VALUES, np.nan, inplace=True)

    # Extra safety: trim whitespace-only values
    df.replace(r"^\s+$", np.nan, regex=True, inplace=True)

    # Optional visibility metric
    total_replaced = df.isna().sum().sum() - raw_df.isna().sum().sum()
    log(f"STEP 4.1 completed | Approx values replaced: {max(total_replaced, 0)}")

    return df


In [ ]:
# ============================================================================
# STEP 4.2: CLEANING FUNCTIONS
# NOTE:
# - This file contains utilities used across multiple steps.
# - Some functions are reused later (country, unit, HS, dates).
# - Execution order must be controlled by the caller.
# ============================================================================

def standardize_company_suffix(name):
    """Standardize company legal suffixes."""
    if pd.isna(name):
        return name

    name = str(name).upper().strip()

    suffix_mappings = [
        (r'\bPVT\.?\s*LTD\.?\s*$', 'PVT LTD'),
        (r'\bPRIVATE\s*LTD\.?\s*$', 'PVT LTD'),
        (r'\bPRIVATE\s*LIMITED\s*$', 'PVT LTD'),
        (r'\bP\.?\s*LTD\.?\s*$', 'PVT LTD'),
        (r'\bLIMITED\s*$', 'LIMITED'),
        (r'\bLTD\.?\s*$', 'LIMITED'),
        (r'\bCORPORATION\s*$', 'CORP'),
        (r'\bCORP\.?\s*$', 'CORP'),
        (r'\bINCORPORATED\s*$', 'INC'),
        (r'\bINC\.?\s*$', 'INC'),
        (r'\bCOMPANY\s*$', 'CO'),
        (r'\bCO\.?\s*$', 'CO'),
        (r'\bL\.?L\.?C\.?\s*$', 'LLC'),
        (r'\bGMBH\s*$', 'GMBH'),
    ]

    for pattern, replacement in suffix_mappings:
        name = re.sub(pattern, replacement, name)

    return name


def clean_company_name(name):
    """Clean company name - remove extra spaces, special chars."""
    if pd.isna(name):
        return name

    name = str(name).upper().strip()
    name = re.sub(r'[^A-Z0-9\s&.-]', ' ', name)
    name = re.sub(r'([A-Z])-([A-Z])', r'\1 \2', name)
    name = re.sub(r'\s+', ' ', name)

    return name.strip()


def remove_common_words(name):
    """Remove common filler prefixes."""
    if pd.isna(name):
        return name

    name = str(name).upper()

    remove_patterns = [
        r'^THE\s+',
        r'^M/S\s+',
        r'^M\.?S\.?\s+',
        r'^MESSRS\.?\s+',
        r'\s+&\s+ASSOCIATES$',
        r'\s+AND\s+CO$'
    ]

    for pattern in remove_patterns:
        name = re.sub(pattern, '', name)

    return name.strip()


def standardize_company_name_full(name):
    """Full company name standardization pipeline."""
    if pd.isna(name):
        return name

    name = clean_company_name(name)
    name = standardize_company_suffix(name)
    name = remove_common_words(name)

    return re.sub(r'\s+', ' ', name).strip()


def normalize_country(x):
    """
    Normalize country name using mapping.
    IMPORTANT:
    - COUNTRY_MAPPING must be loaded BEFORE calling this
    - If not present, function will safely return cleaned value
    """
    if pd.isna(x):
        return np.nan

    x = str(x).upper().strip()
    x = re.sub(r"[.,''-]", "", x)
    x = re.sub(r"\s+", " ", x)

    # 🔒 Guard: mapping may not be loaded in early steps
    if "COUNTRY_MAPPING" not in globals():
        return x

    for canonical, variants in COUNTRY_MAPPING.items():
        if x in [v.upper() for v in variants]:
            return canonical

    return x


def clean_and_prepare(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply cleaning steps.

    NOTE ON FLOW:
    - In Colab demo: you may only rely on importer/exporter + basic stats
    - HS filtering (6815) should ideally happen BEFORE heavy cleaning
    - Caller controls which columns exist at runtime
    """

    df = df.copy()

    # ------------------------------------------------------------------------
    # Global error value cleanup (safe to run early)
    # ------------------------------------------------------------------------
    df = df.replace(ERROR_VALUES, np.nan)
    df = df.replace("", np.nan)

    # ------------------------------------------------------------------------
    # Importer / Exporter cleaning (CORE FOR STEP 4.2)
    # ------------------------------------------------------------------------
    for col in ["IMPORTER_NAME", "EXPORTER_NAME"]:
        if col in df.columns:
            df[col] = df[col].apply(standardize_company_name_full)

    # ------------------------------------------------------------------------
    # Country normalization (optional at this stage)
    # ------------------------------------------------------------------------
    if "COUNTRY" in df.columns:
        df["COUNTRY"] = df["COUNTRY"].apply(normalize_country)

    # ------------------------------------------------------------------------
    # Unit normalization (depends on UNIT_CORRECTIONS availability)
    # ------------------------------------------------------------------------
    if "UNIT" in df.columns:
        df["UNIT"] = (
            df["UNIT"]
            .astype(str)
            .str.upper()
            .str.strip()
            .replace(UNIT_CORRECTIONS)
            .replace(ERROR_VALUES, np.nan)
        )

    # ------------------------------------------------------------------------
    # Numeric fields
    # ------------------------------------------------------------------------
    if "FOB_VALUE" in df.columns:
        df["FOB_VALUE"] = pd.to_numeric(df["FOB_VALUE"], errors="coerce")

    if "QUANTITY" in df.columns:
        df["QUANTITY"] = pd.to_numeric(df["QUANTITY"], errors="coerce")

    # ------------------------------------------------------------------------
    # Date handling (may be skipped during demo runs)
    # ------------------------------------------------------------------------
    if "SB_DATE" in df.columns:
        df["SB_DATE"] = pd.to_datetime(
            df["SB_DATE"],
            errors="coerce",
            infer_datetime_format=True
        )
        df["YEAR"] = df["SB_DATE"].dt.year
        df["MONTH"] = df["SB_DATE"].dt.month
        df["WEEK"] = df["SB_DATE"].dt.isocalendar().week

    # ------------------------------------------------------------------------
    # HS derivations
    # NOTE:
    # - HS 6815 filtering should ideally occur BEFORE calling this function
    # ------------------------------------------------------------------------
    if "HS_CODE" in df.columns:
        df["HS_CODE"] = df["HS_CODE"].astype(str).str.replace(r"\D", "", regex=True)
        df["HS_2"] = df["HS_CODE"].str[:2]
        df["HS_4"] = df["HS_CODE"].str[:4]
        df["HS_6"] = df["HS_CODE"].str[:6]

    return df


In [ ]:
# ============================================================================
# STEP 5: DEDUPLICATION
# ============================================================================

from typing import Tuple

def deduplicate_smart(df: pd.DataFrame) -> Tuple[pd.DataFrame, dict]:
    """
    Deduplicate using priority:
    1. SB_NO (shipping bill is official document)
    2. INV_NO (invoice number as fallback)
    3. Full row duplicate

    NOTE:
    - Columns may not exist in all monthly files
    - Function must be safe for partial data
    """

    df = df.copy()

    stats = {}
    stats["before"] = len(df)

    # ------------------------------------------------------------------------
    # Priority 1: Shipping Bill Number
    # ------------------------------------------------------------------------
    if "SB_NO" in df.columns and df["SB_NO"].notna().any():
        df = df.drop_duplicates(subset=["SB_NO"], keep="first")
        stats["method"] = "SB_NO"

    # ------------------------------------------------------------------------
    # Priority 2: Invoice Number
    # ------------------------------------------------------------------------
    elif "INV_NO" in df.columns and df["INV_NO"].notna().any():
        df = df.drop_duplicates(subset=["INV_NO"], keep="first")
        stats["method"] = "INV_NO"

    # ------------------------------------------------------------------------
    # Priority 3: Full-row duplicate
    # ------------------------------------------------------------------------
    else:
        df = df.drop_duplicates(keep="first")
        stats["method"] = "FULL_ROW"

    stats["after"] = len(df)
    stats["removed"] = stats["before"] - stats["after"]

    return df, stats


In [ ]:
# ============================================================================
# STEP 6: SANITY CHECKS & REPORTING
# ============================================================================

def print_sanity_check(raw_stats: dict, cleaned_stats: dict):
    """Print before/after comparison with safe guards."""

    def safe_get(d, key, default="NA"):
        return d.get(key, default) if isinstance(d, dict) else default

    def safe_num(x):
        return x if pd.notna(x) else 0

    print("\n" + "=" * 70)
    print("SANITY CHECK: BEFORE vs AFTER CLEANING")
    print("=" * 70)

    print("\n📊 Row count:")
    print(f"  Before: {safe_get(raw_stats, 'total_rows')}")
    print(f"  After:  {safe_get(cleaned_stats, 'total_rows')}")

    print("\n💰 Average FOB Value (INR):")
    try:
        print(f"  Before: {safe_num(safe_get(raw_stats, 'avg_fob')):,.2f}")
    except Exception:
        print(f"  Before: {safe_get(raw_stats, 'avg_fob')}")

    try:
        print(f"  After:  {safe_num(safe_get(cleaned_stats, 'avg_fob')):,.2f}")
    except Exception:
        print(f"  After:  {safe_get(cleaned_stats, 'avg_fob')}")

    print("\n🏢 Distinct Importers:")
    print(f"  Before: {safe_get(raw_stats, 'distinct_importers')}")
    print(f"  After:  {safe_get(cleaned_stats, 'distinct_importers')}")

    print("\n🌍 Distinct Countries:")
    print(f"  Before: {safe_get(raw_stats, 'distinct_countries')}")
    print(f"  After:  {safe_get(cleaned_stats, 'distinct_countries')}")

    print("\n" + "=" * 70)


def contact_coverage_report(df: pd.DataFrame):
    """Report on importer contact/address availability."""

    print("\n" + "=" * 70)
    print("CONTACT INFORMATION AVAILABILITY")
    print("=" * 70)

    if df is None or df.empty:
        print("No data available for contact coverage analysis.")
        print("=" * 70)
        return

    if "IMPORTER_CONTACT" in df.columns:
        coverage = df["IMPORTER_CONTACT"].notna().mean() * 100
        print(f"Importer contact available: {coverage:.1f}%")
    else:
        print("Importer contact column not present.")

    if "IMPORTER_ADDRESS" in df.columns:
        coverage = df["IMPORTER_ADDRESS"].notna().mean() * 100
        print(f"Importer address available: {coverage:.1f}%")
    else:
        print("Importer address column not present.")

    print("=" * 70)


In [ ]:
# ============================================================================
# STEP 7: ANALYTICS
# ============================================================================

def analyze_top_importers(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    """Get top importers by FOB value."""

    df = df.copy()

    # Guard required columns
    required_cols = ["IMPORTER_NAME", "FOB_VALUE"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Required column missing for analysis: {col}")

    top_importers = (
        df
        .groupby(["IMPORTER_NAME"])
        .agg(
            total_fob_inr=("FOB_VALUE", "sum"),
            total_quantity=("QUANTITY", "sum") if "QUANTITY" in df.columns else ("FOB_VALUE", "size"),
            num_shipments=("SB_NO", "nunique") if "SB_NO" in df.columns else ("FOB_VALUE", "size"),
            num_invoices=("INV_NO", "nunique") if "INV_NO" in df.columns else ("FOB_VALUE", "size"),
            countries=("COUNTRY", "nunique") if "COUNTRY" in df.columns else ("FOB_VALUE", "size"),
            avg_fob_per_shipment=("FOB_VALUE", "mean"),
        )
        .reset_index()
        .sort_values("total_fob_inr", ascending=False)
        .head(top_n)
    )

    return top_importers


def analyze_by_country(df: pd.DataFrame) -> pd.DataFrame:
    """Analyze by destination country."""

    df = df.copy()

    if "COUNTRY" not in df.columns:
        raise ValueError("COUNTRY column required for country-level analysis")

    by_country = (
        df
        .groupby(["COUNTRY"])
        .agg(
            total_fob_inr=("FOB_VALUE", "sum"),
            num_shipments=("SB_NO", "nunique") if "SB_NO" in df.columns else ("FOB_VALUE", "size"),
            num_importers=("IMPORTER_NAME", "nunique") if "IMPORTER_NAME" in df.columns else ("FOB_VALUE", "size"),
            avg_fob_per_shipment=("FOB_VALUE", "mean"),
        )
        .reset_index()
        .sort_values("total_fob_inr", ascending=False)
    )

    return by_country


def analyze_gulf_middle_east(
    df: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Filter for:
    - Combined Gulf + Middle East
    - Gulf only
    - Middle East only
    """

    df = df.copy()

    if "COUNTRY" not in df.columns:
        raise ValueError("COUNTRY column required for Gulf / Middle East analysis")

    # Guard mappings
    if not all(name in globals() for name in ["GULF_COUNTRIES", "MIDDLE_EAST_COUNTRIES", "GULF_MIDDLE_EAST"]):
        raise ValueError("Gulf / Middle East country mappings not loaded")

    gulf_df = df[df["COUNTRY"].isin(GULF_COUNTRIES)].copy()
    me_df = df[df["COUNTRY"].isin(MIDDLE_EAST_COUNTRIES)].copy()
    gme_df = df[df["COUNTRY"].isin(GULF_MIDDLE_EAST)].copy()

    return gme_df, gulf_df, me_df


In [ ]:
# ============================================================================
# STEP 8: EXPORT & DOWNLOAD
# ============================================================================

def export_results(df: pd.DataFrame, top_importers: pd.DataFrame,
                   by_country: pd.DataFrame, gme_df: pd.DataFrame):
    """Export all results as CSV."""

    df.to_csv("hs6815_cleaned_full.csv", index=False)
    if "log" in globals():
        log("Exported: hs6815_cleaned_full.csv")

    top_importers.to_csv("hs6815_top_importers.csv", index=False)
    if "log" in globals():
        log("Exported: hs6815_top_importers.csv")

    by_country.to_csv("hs6815_by_country.csv", index=False)
    if "log" in globals():
        log("Exported: hs6815_by_country.csv")

    gme_df.to_csv("hs6815_gulf_middle_east.csv", index=False)
    if "log" in globals():
        log("Exported: hs6815_gulf_middle_east.csv")


def download_results():
    """Trigger download of all CSVs."""
    if IN_COLAB:
        files.download("hs6815_cleaned_full.csv")
        files.download("hs6815_top_importers.csv")
        files.download("hs6815_by_country.csv")
        files.download("hs6815_gulf_middle_east.csv")
        if "log" in globals():
            log("All files ready for download.")
    else:
        print("Download skipped (not running in Colab).")

# ============================================================
# MAIN PIPELINE FUNCTION
# ============================================================

def main():
    """
    Runs the complete EXIM pipeline:
    1. Upload files (STEP 1)
    2. Ingest, map, clean HS codes (STEP 2)
    3. Baseline raw stats (STEP 3)
    4. Global error replacement (STEP 4.1)
    5. Advanced analytics & insights
    """

    log("Starting EXIM pipeline...")

    # ----------------------------
    # STEP 1: Upload files
    # ----------------------------
    uploaded_files, valid_files_list = upload_files()

    # ----------------------------
    # STEP 2: Read, map, clean, append
    # ----------------------------
    processed_dfs = []

    for fname in valid_files_list:
        log(f"Processing file: {fname}")
        try:
            if fname.lower().endswith((".xlsx", ".xls")):
                df = pd.read_excel(fname)
            elif fname.lower().endswith(".csv"):
                df = pd.read_csv(fname)
        except Exception as e:
            log(f"Failed to read {fname}: {e}")
            continue

        if df.empty:
            log(f"Empty file skipped: {fname}")
            continue

        df = standardize_columns(df)
        mapped_df = map_columns(df, COLUMN_SELECTION_MAP)
        mapped_df["SOURCE_FILE"] = fname
        mapped_df["HS_CODE"] = mapped_df["HS_CODE"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
        mapped_df = mapped_df[mapped_df["HS_CODE"].str.startswith(TARGET_HS_PREFIX, na=False)]
        if mapped_df.empty:
            log(f"No relevant HS data in {fname}")
            continue
        processed_dfs.append(mapped_df)

    if not processed_dfs:
        raise ValueError("No data found for the target HS code across uploaded files.")

    raw_df = pd.concat(processed_dfs, ignore_index=True)
    log(f"STEP 2 completed: Total rows = {len(raw_df)}")

    # ----------------------------
    # STEP 3: Raw data statistics
    # ----------------------------
    log("STEP 3: Raw data statistics")
    raw_stats = {}
    raw_stats["total_rows"] = len(raw_df)
    CRITICAL_COLUMNS = ["HS_CODE", "IMPORTER_NAME", "COUNTRY", "FOB_INR", "QUANTITY", "UNIT"]
    raw_stats["missing_percentage"] = (raw_df[CRITICAL_COLUMNS].isna().mean()*100).round(2).to_dict()
    raw_stats["avg_fob"] = pd.to_numeric(raw_df["FOB_INR"], errors="coerce").mean()
    raw_stats["median_fob"] = pd.to_numeric(raw_df["FOB_INR"], errors="coerce").median()
    raw_stats["distinct_importers"] = raw_df["IMPORTER_NAME"].nunique(dropna=True)
    raw_stats["distinct_countries"] = raw_df["COUNTRY"].nunique(dropna=True)
    raw_stats["distinct_units"] = raw_df["UNIT"].nunique(dropna=True)

    log("Raw stats snapshot:")
    for k,v in raw_stats.items():
        print(f"{k}: {v}")

    # ----------------------------
    # STEP 4.1: Global error replacement
    # ----------------------------
    raw_df = step_4_1_global_error_replacement(raw_df)

    # ----------------------------
    # STEP 5: Advanced analytics
    # ----------------------------
    log("STEP 5: Generating advanced analytics")

    df_analysis = raw_df.copy()
    # Ensure numeric columns
    df_analysis["FOB_INR"] = pd.to_numeric(df_analysis["FOB_INR"], errors="coerce")
    df_analysis["FOB_FC"] = pd.to_numeric(df_analysis["FOB_FC"], errors="coerce")
    df_analysis["QUANTITY"] = pd.to_numeric(df_analysis["QUANTITY"], errors="coerce")

    # ---- Derived metric: price per unit INR / FC
    df_analysis["PRICE_PER_UNIT_INR"] = df_analysis["FOB_INR"] / df_analysis["QUANTITY"]
    df_analysis["PRICE_PER_UNIT_FC"] = df_analysis["FOB_FC"] / df_analysis["QUANTITY"]

    # ---- Top countries by avg quantity and avg FOB
    country_stats = (
        df_analysis.groupby("COUNTRY")
        .agg(
            avg_quantity=("QUANTITY", "mean"),
            avg_fob=("FOB_INR", "mean"),
            total_fob=("FOB_INR", "sum"),
            total_transactions=("FOB_INR", "count")
        )
        .sort_values("total_fob", ascending=False)
    )

    print("\nTop countries by total FOB:")
    print(country_stats.head(10))

    # ---- Top importers globally
    importer_stats = (
        df_analysis.groupby("IMPORTER_NAME")
        .agg(
            total_fob=("FOB_INR", "sum"),
            avg_fob=("FOB_INR", "mean"),
            total_quantity=("QUANTITY", "sum"),
            transaction_count=("FOB_INR", "count")
        )
        .sort_values("total_fob", ascending=False)
    )

    print("\nTop importers globally:")
    print(importer_stats.head(10))

    # ---- Steady importers: present in multiple files / months
    steady_importers = (
        df_analysis.groupby("IMPORTER_NAME")["SOURCE_FILE"]
        .nunique()
        .sort_values(ascending=False)
    )
    print("\nImporters with steady presence (by number of files):")
    print(steady_importers.head(15))

    # ---- Mode of import analysis: total FOB / avg FOB per transaction
    mode_stats = (
        df_analysis.groupby("MODE_OF_PORT")
        .agg(
            total_fob=("FOB_INR", "sum"),
            avg_fob_per_transaction=("FOB_INR", "mean"),
            transaction_count=("FOB_INR", "count")
        )
        .sort_values("total_fob", ascending=False)
    )
    print("\nMode of import statistics:")
    print(mode_stats)

    # ---- Gulf & Middle East importers
    gulf_me_importers = df_analysis[
        df_analysis["COUNTRY"].isin(GULF_MIDDLE_EAST)
    ].groupby("IMPORTER_NAME").agg(
        total_fob=("FOB_INR", "sum"),
        total_quantity=("QUANTITY", "sum"),
        transaction_count=("FOB_INR", "count"),
        country=("COUNTRY", "first")
    ).sort_values("total_fob", ascending=False)

    print("\nGulf & Middle East importers:")
    print(gulf_me_importers.head(20))

    # ---- Additional insights: units & HS codes
    print("\nTop HS codes by total FOB:")
    print(df_analysis.groupby("HS_CODE")["FOB_INR"].sum().sort_values(ascending=False).head(10))

    print("\nTop units by quantity:")
    print(df_analysis.groupby("UNIT")["QUANTITY"].sum().sort_values(ascending=False).head(10))

    log("Pipeline execution completed successfully.")

    # Return final DataFrames for further exploration or export
    return {
        "raw_df": raw_df,
        "country_stats": country_stats,
        "importer_stats": importer_stats,
        "steady_importers": steady_importers,
        "mode_stats": mode_stats,
        "gulf_me_importers": gulf_me_importers
    }


# V.0.0.1 Analytics code with report generation in form of excel

In [ ]:
# ============================================================
# HS 6815 GULF & MIDDLE EAST IMPORTER INTELLIGENCE SYSTEM
# FINAL VERSION - HS Codes + Surface Importer Cleaning
# ============================================================

import os
import sys
import warnings
import re
from typing import List, Dict, Tuple
from datetime import datetime

# Silence warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("\n" + "="*70)
print("STEP 0: IMPORTS & CONFIGURATION")
print("="*70)

# ============================================================
# IMPORTS WITH ERROR HANDLING
# ============================================================

try:
    import pandas as pd
    print("✅ pandas imported")
except ImportError as e:
    raise ImportError("pandas not installed. Run: !pip install pandas") from e

try:
    import numpy as np
    print("✅ numpy imported")
except ImportError as e:
    raise ImportError("numpy not installed. Run: !pip install numpy") from e

try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter
    print("✅ openpyxl imported")
except ImportError:
    print("⚠️  openpyxl not available - installing...")
    os.system("pip install openpyxl -q")
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter
    print("✅ openpyxl installed & imported")

# Colab-specific
try:
    from google.colab import files
    IN_COLAB = True
    print("✅ Google Colab detected")
except ImportError:
    IN_COLAB = False
    print("⚠️  Not in Colab (local mode)")


# ============================================================
# PANDAS CONFIGURATION
# ============================================================

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_rows", 1000)
pd.options.mode.chained_assignment = None

print("✅ Pandas configured")


# ============================================================
# GLOBAL CONSTANTS
# ============================================================

TARGET_HS_PREFIX = "6815"
ALLOWED_FILE_EXTENSIONS = (".xlsx", ".xls", ".csv")

print(f"✅ Configuration: HS {TARGET_HS_PREFIX}")


# ============================================================
# ERROR VALUES & STANDARDIZATIONS
# ============================================================

ERROR_VALUES = [
    "#REF!", "#NAME?", "#N/A", "#VALUE!", "#DIV/0!", "#NULL!",
    "nana", "NA", "N/A", "Na", "N A", "N?A", "n/a",
    "Null", "NULL", "null", "UNKNOWN", "Unknown", "unknown",
    "NOT FOUND", "Not Found", "not found",
    "0000", "00000000", "000000",
    "NIL", "Nil", "nil", "NONE", "None", "none",
    "-", "--", "---", "____", "X", "XX", "XXX"
]

UNIT_CORRECTIONS = {
    'KGA': 'KGS', 'KILO': 'KGS', 'KG': 'KGS', 'KILOGRAMS': 'KGS',
    'MET': 'MTR', 'METER': 'MTR', 'METRES': 'MTR', 'METERS': 'MTR',
    'TONNE': 'MTS', 'TON': 'MTS', 'MT': 'MTS', 'TONS': 'MTS',
    'PIECE': 'PCS', 'PC': 'PCS', 'PIECES': 'PCS',
    'NUMBER': 'NOS', 'NO': 'NOS', 'NUM': 'NOS', 'NUMBERS': 'NOS',
    'DOZEN': 'DOZ', 'DZ': 'DOZ', 'PAIR': 'PRS', 'PAIRS': 'PRS',
    'SETS': 'SET', 'SQM': 'SQM', 'SQMTR': 'SQM', 'SQF': 'SQF', 'SQFT': 'SQF',
    'GROSS': 'GRS', 'FEET': 'FTS', 'FT': 'FTS',
    'CENTIMETER': 'CMS', 'CM': 'CMS', 'KILOMETER': 'KME', 'KM': 'KME',
    'CARTON': 'CTN', 'CTNS': 'CTN', 'LITER': 'LTR', 'LITRE': 'LTR', 'LTS': 'LTR'
}

COUNTRY_MAPPING = {
    "UNITED ARAB EMIRATES": ["UNITED ARAB EMIRATES", "UAE", "UNITED ARAB E"],
    "SAUDI ARABIA": ["SAUDI ARABIA", "KSA"],
    "QATAR": ["QATAR"],
    "KUWAIT": ["KUWAIT"],
    "OMAN": ["OMAN"],
    "BAHRAIN": ["BAHRAIN"],
    "IRAN": ["IRAN", "IRAN ISLAMIC REP"],
    "IRAQ": ["IRAQ"],
    "JORDAN": ["JORDAN"],
    "ISRAEL": ["ISRAEL"],
    "TURKEY": ["TURKEY", "TURKIYE"],
    "EGYPT": ["EGYPT"],
    "LEBANON": ["LEBANON"],
    "YEMEN": ["YEMEN", "YEMEN DEMOCRATIC"],
    "UNITED STATES": ["UNITED STATES", "USA", "U.S.A"],
    "UNITED KINGDOM": ["UNITED KINGDOM", "UK"],
    "CHINA": ["CHINA"],
    "HONG KONG": ["HONG KONG", "HONGKONG"],
    "INDIA": ["INDIA"],
    "SINGAPORE": ["SINGAPORE"],
    "MALAYSIA": ["MALAYSIA"],
    "INDONESIA": ["INDONESIA"],
    "THAILAND": ["THAILAND"],
    "VIETNAM": ["VIETNAM"],
    "GERMANY": ["GERMANY"],
    "FRANCE": ["FRANCE"],
    "ITALY": ["ITALY"],
    "SPAIN": ["SPAIN"],
    "NETHERLANDS": ["NETHERLANDS"],
    "BELGIUM": ["BELGIUM"],
    "AUSTRIA": ["AUSTRIA"],
    "SWITZERLAND": ["SWITZERLAND"],
    "SWEDEN": ["SWEDEN"],
    "NORWAY": ["NORWAY"],
    "DENMARK": ["DENMARK"],
    "POLAND": ["POLAND"],
    "RUSSIA": ["RUSSIA", "RUSSIAN FEDERATION"],
    "JAPAN": ["JAPAN"],
    "SOUTH KOREA": ["SOUTH KOREA", "KOREA REPUBLIC OF"],
    "BRAZIL": ["BRAZIL"],
    "MEXICO": ["MEXICO"],
    "CANADA": ["CANADA"],
    "AUSTRALIA": ["AUSTRALIA"],
    "NEW ZEALAND": ["NEW ZEALAND"],
}

GULF_COUNTRIES = {
    "UNITED ARAB EMIRATES", "SAUDI ARABIA", "QATAR",
    "KUWAIT", "OMAN", "BAHRAIN",
}

MIDDLE_EAST_COUNTRIES = {
    "IRAN", "IRAQ", "JORDAN", "ISRAEL", "TURKEY",
    "EGYPT", "LEBANON", "YEMEN",
}

GULF_MIDDLE_EAST = GULF_COUNTRIES | MIDDLE_EAST_COUNTRIES

print(f"✅ Mappings loaded")


# ============================================================
# CANONICAL COLUMN MAP
# ============================================================

COLUMN_SELECTION_MAP: Dict[str, List[str]] = {
    "HS_CODE": [
        "HS CODE", "HS_CODE", "HSN_CODE", "RITC", "HSCODE",
        "HSN CODE", "HSN", "TARIFF CODE", "HS", "RITCCODE", "RITC_8", "RITC_CODE"
    ],
    "PRODUCT_DESCRIPTION": [
        "PRODUCT DESCRIPTION", "ITEM", "ITEM DESCRIPTION", "DESCRIPTION",
        "GOODS DESCRIPTION", "COMMODITY", "PRODUCT"
    ],
    "QUANTITY": ["QUANTITY", "PRODUCT_QUANTITY", "QTY", "QTY."],
    "UNIT": [
        "UQC", "UNIT", "QUANTITY_UNIT", "UNIT QUANTITY", "UOM",
        "UNIT OF MEASURE", "QUANTITY UNIT"
    ],
    "UNIT_RATE_INR": [
        "UNIT RATE IN INR", "UNIT PRICE IN INR", "RATE INR", "PER UNIT FOB",
        "UNT PRICE INR", "UNIT_VALUE_INR", "UNIT_VALUE_IN_INR", "UNIT_RATE_IN_INR"
    ],
    "UNIT_RATE_FC": [
        "ITEM_RATE", "UNIT RATE IN FC", "UNT PRICE FC", "UNIT PRICE (USD)",
        "UNIT RATE", "UNIT PRICE", "RATE", "UNIT_VALUE_USD", "UNIT_VALUE_FC",
        "UNIT_RATE_USD", "UNITPRICE", "UNIT PRICE FOREIGN", "UNIT PRICE FC",
        "ITEM_RATE_IN_FC", "UNIT RATE IN FOREIGN CURRENCY"
    ],
    "FOB_INR": [
        "FOB", "FOB INR", "FOB IN INR", "FOB VALUE (INR)", "FOB_IN_INR",
        "FOB VALUE", "FOB_VALUE", "VALUE INR", "INR VALUE", "FOB IN INR.1",
        "FOBVALUEINRS", "TOTAL_VALUE_IN_INR", "TOTAL FOB VALUE IN INR"
    ],
    "FOB_FC": [
        "FOB IN FC", "INV VALUE FC", "FOB FC", "VALUE IN FC",
        "FOB USD", "USD VALUE", "FC VALUE", "TOTAL VALUE IN FC",
        "TOTAL_VALUE_IN_FC", "TOTAL_VALUE_USD", "TOTAL_VALUE_FC", "TOTAL_VALUE_IN_USD"
    ],
    "CURRENCY": [
        "CURRENCY", "CURR", "CUR", "CURRENCY CODE", "CURRENCY_NAME", "UNIT RATE CURRENCY"
    ],
    "EXPORTER_NAME": [
        "EXPORTER", "EXPORTER NAME", "EXPORTER NAMES", "SHIPPER",
        "SHIPPER NAME", "SUPPLIER", "SELLER", "INDIAN EXPORTER NAME"
    ],
    "EXPORTER_ID": [
        "EXPORTER ID", "IEC", "IEC CODE", "IEC NO",
        "IE CODE", "IMPORTER EXPORTER CODE", "IECNO", "IEC_NO"
    ],
    "EXPORTER_ADDRESS": [
        "EXPORTER ADDRESS", "EXPORTER ADD", "SHIPPER ADDRESS"
    ],
    "IMPORTER_NAME": [
        "IMPORTER NAME", "IMPORTER NAMES", "IMPORTER_NAME", "CONSIGNEE",
        "IMPORTER", "BUYER", "BUYER NAME", "CONSIGNEE NAME",
        "CONSINEENAME", "CONSIGNEENAME", "CONSINEE_NAME", "CONSIGNEE_NAME"
    ],
    "IMPORTER_ADDRESS": [
        "IMPORTER ADDRESS", "Consignee_Address", "BUYER ADDRESS", "CONSIGNEE_ADDRESS"
    ],
    "IMPORTER_CONTACT": [
        "IMPORTER CONTACT", "PHONE", "EMAIL", "MOBILE", "CONTACT"
    ],
    "INDIAN_PORT": [
        "PORT CODE", "ORIGIN PORT", "INDIAN PORT", "PORT",
        "LOADING PORT", "PORT OF LOADING", "PORT OF ORIGIN"
    ],
    "FOREIGN_PORT": [
        "FOREIGN PORT", "DESTINATION PORT", "DISCHARGE PORT", "POD"
    ],
    "COUNTRY": [
        "COUNTRY", "FOREIGN COUNTRY", "DESTINATION_COUNTRY", "DEST COUNTRY",
        "DESTINATION", "COUNTRY OF DESTINATION", "CONSIGNEE COUNTRY"
    ],
    "SB_DATE": [
        "SBDT", "SB DATE", "SBDATE", "SHIPPING_DATE", "SHIPPING BILL DATE",
        "SHIPPING DATE", "DATE", "EXPORT DATE", "SB_DATE", "SB_DT"
    ],
    "SB_NO": [
        "SBNO", "SB NO", "SHIPPING BILL NO", "SHIPPING_BILL_NO",
        "SB NUMBER", "BILL NO", "SBNUMBER", "SYSTEM_ID", "SB.NO."
    ],
    "INVOICE_NO": [
        "INVOICE_NO", "INVOICE NO", "INV NO", "INVOICE NUMBER", "INVOICE NO."
    ],
}

print(f"✅ Column map configured")

print("\n" + "="*70)
print("✅ STEP 0 COMPLETED")
print("="*70 + "\n")


# ============================================================
# UTILITY: LOGGING
# ============================================================

def log(message: str, level: str = "INFO") -> None:
    """Safe logging function."""
    emoji_map = {"INFO": "ℹ️ ", "SUCCESS": "✅", "WARNING": "⚠️ ", "ERROR": "❌"}
    emoji = emoji_map.get(level, "→ ")
    print(f"{emoji} {message}")


# ============================================================
# STEP 1: UPLOAD FILES
# ============================================================

def upload_files() -> Tuple[Dict, List[str]]:
    """Upload files via Colab UI."""
    if not IN_COLAB:
        raise EnvironmentError("File upload supported only in Google Colab.")

    log("Please upload monthly import files (XLS / XLSX / CSV).")
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No files uploaded.")

    log(f"Total files uploaded: {len(uploaded)}")

    valid_files = [
        f for f in uploaded.keys()
        if f.lower().endswith(ALLOWED_FILE_EXTENSIONS)
    ]

    if not valid_files:
        raise ValueError("No valid data files found.")

    log(f"Valid data files detected: {len(valid_files)}", "SUCCESS")

    print("\nUploaded Files:")
    print("-" * 50)
    for fname in valid_files:
        size_kb = round(len(uploaded[fname]) / 1024, 2)
        print(f"  {fname:<40} {size_kb:>8} KB")
    print("-" * 50)

    return uploaded, valid_files


# ============================================================
# STEP 2: STANDARDIZE & MAP COLUMNS
# ============================================================

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Uppercase and strip column names."""
    df = df.copy()
    df.columns = df.columns.astype(str).str.upper().str.strip()
    return df


def map_columns(df: pd.DataFrame, mapping: Dict[str, List[str]]) -> pd.DataFrame:
    """Map raw columns to canonical schema."""
    mapped_df = pd.DataFrame(index=df.index)

    for canonical_col, variants in mapping.items():
        found = False
        for v in variants:
            if v in df.columns:
                mapped_df[canonical_col] = df[v]
                found = True
                break

        if not found:
            mapped_df[canonical_col] = np.nan

    return mapped_df


# ============================================================
# STEP 2: INGEST ALL FILES
# ============================================================

def ingest_all_files(uploaded: Dict, valid_files: List[str]) -> pd.DataFrame:
    """Read, map, filter HS code, and append all files."""

    log("Starting STEP 2: File ingestion & column mapping")

    processed_dfs = []

    for fname in valid_files:
        log(f"Processing file: {fname}")

        try:
            if fname.lower().endswith((".xlsx", ".xls")):
                df = pd.read_excel(fname)
            elif fname.lower().endswith(".csv"):
                df = pd.read_csv(fname)
            else:
                log(f"Unsupported format: {fname}", "WARNING")
                continue
        except Exception as e:
            log(f"Failed to read {fname}: {e}", "ERROR")
            continue

        if df.empty:
            log(f"Empty file: {fname}", "WARNING")
            continue

        df = standardize_columns(df)
        mapped_df = map_columns(df, COLUMN_SELECTION_MAP)
        mapped_df["SOURCE_FILE"] = fname

        mapped_df["HS_CODE"] = (
            mapped_df["HS_CODE"]
            .astype(str)
            .str.replace(r"\.0$", "", regex=True)
            .str.strip()
        )

        before = len(mapped_df)
        mapped_df = mapped_df[
            mapped_df["HS_CODE"].str.startswith(TARGET_HS_PREFIX, na=False)
        ]
        after = len(mapped_df)

        log(f"  Rows: {before} → {after} (HS {TARGET_HS_PREFIX})")

        if not mapped_df.empty:
            processed_dfs.append(mapped_df)

    if not processed_dfs:
        raise ValueError(f"No HS {TARGET_HS_PREFIX} data found.")

    raw_df = pd.concat(processed_dfs, ignore_index=True)
    log(f"STEP 2 completed: {len(raw_df)} total rows", "SUCCESS")

    return raw_df


# ============================================================
# STEP 3-4: ERROR REPLACEMENT & CLEANING
# ============================================================

def step_4_error_replacement(df: pd.DataFrame) -> pd.DataFrame:
    """Replace error values with NaN."""
    log("Replacing error values with NaN")
    df = df.copy()
    df = df.replace(ERROR_VALUES, np.nan)
    df = df.replace(r"^\s+$", np.nan, regex=True)
    return df


def standardize_company_suffix(name):
    """Standardize company suffixes."""
    if pd.isna(name):
        return name

    name = str(name).upper().strip()

    suffix_mappings = [
        (r'\bPVT\.?\s*LTD\.?\s*$', 'PVT LTD'),
        (r'\bPRIVATE\s*LTD\.?\s*$', 'PVT LTD'),
        (r'\bLIMITED\s*$', 'LIMITED'),
        (r'\bLTD\.?\s*$', 'LIMITED'),
        (r'\bCORPORATION\s*$', 'CORP'),
        (r'\bCORP\.?\s*$', 'CORP'),
        (r'\bINCORPORATED\s*$', 'INC'),
        (r'\bINC\.?\s*$', 'INC'),
        (r'\bCOMPANY\s*$', 'CO'),
        (r'\bCO\.?\s*$', 'CO'),
        (r'\bL\.?L\.?C\.?\s*$', 'LLC'),
        (r'\bGMBH\s*$', 'GMBH'),
    ]

    for pattern, replacement in suffix_mappings:
        name = re.sub(pattern, replacement, name)

    return name


def clean_company_name(name):
    """Clean company name."""
    if pd.isna(name):
        return name

    name = str(name).upper().strip()
    name = re.sub(r'[^A-Z0-9\s&.-]', ' ', name)
    name = re.sub(r'([A-Z])-([A-Z])', r'\1 \2', name)
    name = re.sub(r'\s+', ' ', name)

    return name.strip()


def remove_common_words(name):
    """Remove prefixes."""
    if pd.isna(name):
        return name

    name = str(name).upper()
    patterns = [r'^THE\s+', r'^M/S\s+', r'^M\.?S\.?\s+', r'^MESSRS\.?\s+']

    for pattern in patterns:
        name = re.sub(pattern, '', name)

    return name.strip()


def standardize_company_name_full(name):
    """Full company name standardization."""
    if pd.isna(name):
        return name

    name = clean_company_name(name)
    name = standardize_company_suffix(name)
    name = remove_common_words(name)

    return re.sub(r'\s+', ' ', name).strip()


def clean_importer_name_enhanced(name):
    """
    ENHANCED cleaning for IMPORTER NAMES ONLY
    Includes:
    - Remove unwanted punctuation
    - Replace multiple spaces
    - Handle "TO ORDER" variations
    - Convert symbol-only strings to "TO ORDER"
    """
    if pd.isna(name):
        return name

    name = str(name).upper().strip()

    # Step 1: Standard cleaning
    name = clean_company_name(name)
    name = standardize_company_suffix(name)
    name = remove_common_words(name)

    # Step 2: Remove unwanted punctuation (enhanced for importers)
    name = re.sub(r"[.,''()/]", "", name)

    # Step 3: Replace multiple spaces with single space
    name = re.sub(r"\s+", " ", name)

    # Step 4: Check if only symbols (convert to "TO ORDER")
    if re.match(r"^[^\w]*$", name):
        return "TO ORDER"

    # Step 5: Normalize variations of "TO ORDER"
    name = re.sub(r"TO\s+(THE\s+)?ORDER(\s+OF.*)?", "TO ORDER", name)

    return name.strip()


def normalize_country(x):
    """Normalize country name."""
    if pd.isna(x):
        return np.nan

    x = str(x).upper().strip()
    x = re.sub(r"[.,''-]", "", x)
    x = re.sub(r"\s+", " ", x)

    for canonical, variants in COUNTRY_MAPPING.items():
        if x in [v.upper() for v in variants]:
            return canonical

    return x


def clean_and_prepare(df: pd.DataFrame) -> pd.DataFrame:
    """Apply all cleaning steps."""

    df = df.copy()

    # Error replacement
    df = df.replace(ERROR_VALUES, np.nan)
    df = df.replace("", np.nan)

    # ===== IMPORTER CLEANING (Enhanced) =====
    if "IMPORTER_NAME" in df.columns:
        df["IMPORTER_NAME"] = df["IMPORTER_NAME"].apply(clean_importer_name_enhanced)

    # ===== EXPORTER CLEANING (Standard) =====
    if "EXPORTER_NAME" in df.columns:
        df["EXPORTER_NAME"] = df["EXPORTER_NAME"].apply(standardize_company_name_full)

    # Country
    if "COUNTRY" in df.columns:
        df["COUNTRY"] = df["COUNTRY"].apply(normalize_country)

    # Unit
    if "UNIT" in df.columns:
        df["UNIT"] = (
            df["UNIT"]
            .astype(str)
            .str.upper()
            .str.strip()
            .replace(UNIT_CORRECTIONS)
            .replace(ERROR_VALUES, np.nan)
        )

    # Numeric fields
    for col in ["FOB_INR", "FOB_FC", "UNIT_RATE_INR", "UNIT_RATE_FC", "QUANTITY"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Dates
    if "SB_DATE" in df.columns:
        df["SB_DATE"] = pd.to_datetime(df["SB_DATE"], errors="coerce", infer_datetime_format=True)
        df["YEAR"] = df["SB_DATE"].dt.year
        df["MONTH"] = df["SB_DATE"].dt.month
        df["WEEK"] = df["SB_DATE"].dt.isocalendar().week

    # HS codes
    if "HS_CODE" in df.columns:
        df["HS_CODE"] = df["HS_CODE"].astype(str).str.replace(r"\D", "", regex=True)
        df["HS_2"] = df["HS_CODE"].str[:2]
        df["HS_4"] = df["HS_CODE"].str[:4]
        df["HS_6"] = df["HS_CODE"].str[:6]

    return df


# ============================================================
# UNIT PRICE CALCULATION
# ============================================================

def calculate_unit_prices(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate or use existing unit prices."""

    df = df.copy()

    log("Calculating unit prices (with fallback logic)", "INFO")

    # UNIT PRICE IN INR
    if "UNIT_RATE_INR" not in df.columns:
        df["UNIT_RATE_INR"] = np.nan

    missing_inr = df["UNIT_RATE_INR"].isna()
    if missing_inr.any() and "FOB_INR" in df.columns and "QUANTITY" in df.columns:
        valid_calc = missing_inr & (df["FOB_INR"] > 0) & (df["QUANTITY"] > 0)
        df.loc[valid_calc, "UNIT_RATE_INR"] = (
            df.loc[valid_calc, "FOB_INR"] / df.loc[valid_calc, "QUANTITY"]
        )

    # UNIT PRICE IN FC
    if "UNIT_RATE_FC" not in df.columns:
        df["UNIT_RATE_FC"] = np.nan

    missing_fc = df["UNIT_RATE_FC"].isna()
    if missing_fc.any() and "FOB_FC" in df.columns and "QUANTITY" in df.columns:
        valid_calc = missing_fc & (df["FOB_FC"] > 0) & (df["QUANTITY"] > 0)
        df.loc[valid_calc, "UNIT_RATE_FC"] = (
            df.loc[valid_calc, "FOB_FC"] / df.loc[valid_calc, "QUANTITY"]
        )

    return df


# ============================================================
# STEP 5: DEDUPLICATION
# ============================================================

def deduplicate_smart(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    """Deduplicate with priority."""

    df = df.copy()
    stats = {"before": len(df)}

    if "SB_NO" in df.columns and df["SB_NO"].notna().any():
        df = df.drop_duplicates(subset=["SB_NO"], keep="first")
        stats["method"] = "SB_NO"
    elif "INV_NO" in df.columns and df["INV_NO"].notna().any():
        df = df.drop_duplicates(subset=["INV_NO"], keep="first")
        stats["method"] = "INV_NO"
    else:
        df = df.drop_duplicates(keep="first")
        stats["method"] = "FULL_ROW"

    stats["after"] = len(df)
    stats["removed"] = stats["before"] - stats["after"]

    return df, stats


# ============================================================
# IMPORTER INTELLIGENCE METRICS
# ============================================================

def calculate_importer_intelligence(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate behavioral metrics with HS codes."""

    log("Calculating importer intelligence metrics", "INFO")

    if "IMPORTER_NAME" not in df.columns:
        log("IMPORTER_NAME column not found", "WARNING")
        return pd.DataFrame()

    importer_metrics = []

    for importer, group in df.groupby("IMPORTER_NAME"):

        metrics = {
            "IMPORTER_NAME": importer,
            "TOTAL_TRANSACTIONS": len(group),
            "TOTAL_FOB_INR": group["FOB_INR"].sum() if "FOB_INR" in group.columns else 0,
            "AVG_TRANSACTION_VALUE": group["FOB_INR"].mean() if "FOB_INR" in group.columns else 0,
            "TOTAL_QUANTITY": group["QUANTITY"].sum() if "QUANTITY" in group.columns else 0,
        }

        # ===== NEW: HS CODES HANDLED =====
        if "HS_CODE" in group.columns:
            unique_hs = group["HS_CODE"].nunique(dropna=True)
            metrics["UNIQUE_HS_CODES"] = unique_hs
            # Get all unique HS codes for this importer
            hs_codes = ",".join(group["HS_CODE"].dropna().unique())
            metrics["HS_CODES_LIST"] = hs_codes[:100]  # Limit to 100 chars for display

        if "HS_6" in group.columns:
            unique_hs6 = group["HS_6"].nunique(dropna=True)
            metrics["UNIQUE_HS_6_DIGIT"] = unique_hs6

        # Supplier loyalty
        if "EXPORTER_NAME" in group.columns:
            unique_suppliers = group["EXPORTER_NAME"].nunique(dropna=True)
            metrics["UNIQUE_SUPPLIERS"] = unique_suppliers
            metrics["SUPPLIER_LOYALTY_SCORE"] = (
                len(group) / unique_suppliers if unique_suppliers > 0 else 0
            )

        # Frequency & consistency
        if "SB_DATE" in group.columns:
            group_sorted = group.sort_values("SB_DATE")
            date_range = (group_sorted["SB_DATE"].max() - group_sorted["SB_DATE"].min()).days
            metrics["DAYS_ACTIVE"] = date_range
            metrics["TRANSACTION_FREQUENCY_DAYS"] = (
                date_range / len(group) if len(group) > 1 else 0
            )
            metrics["IS_LONG_TERM"] = "Yes" if date_range >= 730 else "No"

        # Countries
        if "COUNTRY" in group.columns:
            metrics["COUNTRIES_SERVED"] = group["COUNTRY"].nunique(dropna=True)

        # Contact
        if "IMPORTER_ADDRESS" in group.columns:
            has_address = group["IMPORTER_ADDRESS"].notna().any()
            metrics["HAS_ADDRESS"] = "Yes" if has_address else "No"

        if "IMPORTER_CONTACT" in group.columns:
            has_contact = group["IMPORTER_CONTACT"].notna().any()
            metrics["HAS_CONTACT"] = "Yes" if has_contact else "No"

        # Unit price
        if "UNIT_RATE_INR" in group.columns:
            valid_prices = group["UNIT_RATE_INR"].dropna()
            if len(valid_prices) > 0:
                metrics["AVG_UNIT_PRICE_INR"] = valid_prices.mean()

        # Unit
        if "UNIT" in group.columns:
            primary_unit = group["UNIT"].value_counts().idxmax() if len(group) > 0 else "Unknown"
            metrics["PRIMARY_UNIT"] = primary_unit

        importer_metrics.append(metrics)

    intelligence_df = pd.DataFrame(importer_metrics)
    intelligence_df = intelligence_df.sort_values("TOTAL_FOB_INR", ascending=False)

    log(f"Intelligence calculated for {len(intelligence_df)} importers", "SUCCESS")

    return intelligence_df


# ============================================================
# GULF & MIDDLE EAST ANALYSIS
# ============================================================

def analyze_gulf_middle_east(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Filter Gulf & Middle East."""

    if "COUNTRY" not in df.columns:
        raise ValueError("COUNTRY required")

    gme = df[df["COUNTRY"].isin(GULF_MIDDLE_EAST)].copy()
    gulf = df[df["COUNTRY"].isin(GULF_COUNTRIES)].copy()
    me = df[df["COUNTRY"].isin(MIDDLE_EAST_COUNTRIES)].copy()

    return gme, gulf, me




# ============================================================
# ANALYSIS BY COUNTRY
# ============================================================

def analyze_by_country(df: pd.DataFrame) -> pd.DataFrame:
    """Analyze by country with HS codes."""

    if "COUNTRY" not in df.columns:
        raise ValueError("COUNTRY required")

    agg_dict = {"FOB_INR": "sum"}

    if "SB_NO" in df.columns:
        agg_dict["SB_NO"] = "nunique"
    if "IMPORTER_NAME" in df.columns:
        agg_dict["IMPORTER_NAME"] = "nunique"
    if "HS_CODE" in df.columns:
        agg_dict["HS_CODE"] = "nunique"

    by_country = (
        df
        .groupby("COUNTRY")
        .agg(agg_dict)
        .reset_index()
        .rename(columns={
            "FOB_INR": "total_fob_inr",
            "SB_NO": "num_shipments",
            "IMPORTER_NAME": "num_importers",
            "HS_CODE": "unique_hs_codes"
        })
        .sort_values("total_fob_inr", ascending=False)
    )

    return by_country

# ============================================================
# ADVANCED ANALYTICS FUNCTIONS
# Port & Route Analysis + Unit & Product Trends + Market Metrics
# ============================================================

"""
These functions should be added to FINAL_ENHANCED_PIPELINE_WITH_HS_CODES.py

Add these functions BEFORE the main() function (around line 850)
"""

# ============================================================
# PORT & ROUTE ANALYSIS
# ============================================================

def analyze_port_and_routes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze ports, routes, and transport modes.

    Returns:
    - Indian Port → Destination Country routes
    - FOB value per route
    - High-value vs High-frequency classification
    """

    log("Analyzing ports & routes", "INFO")

    if "INDIAN_PORT" not in df.columns or "COUNTRY" not in df.columns:
        return pd.DataFrame()

    # Route = Indian Port → Destination Country
    route_df = df.groupby(["INDIAN_PORT", "COUNTRY"]).agg({
        "FOB_INR": ["sum", "mean", "count"],
        "SB_NO": "nunique",
        "QUANTITY": "sum"
    }).reset_index()

    route_df.columns = ["INDIAN_PORT", "COUNTRY", "TOTAL_FOB_INR", "AVG_FOB_PER_SHIPMENT",
                        "NUM_SHIPMENTS", "UNIQUE_SHIPMENTS", "TOTAL_QUANTITY"]

    # Add transport mode if available
    if "MODE_OF_PORT" in df.columns:
        mode_dist = df.groupby(["INDIAN_PORT", "COUNTRY"])["MODE_OF_PORT"].apply(
            lambda x: x.value_counts().index[0] if len(x) > 0 else "Unknown"
        ).reset_index()
        mode_dist.columns = ["INDIAN_PORT", "COUNTRY", "PRIMARY_TRANSPORT_MODE"]
        route_df = route_df.merge(mode_dist, on=["INDIAN_PORT", "COUNTRY"], how="left")

    # Categorize: High-value vs High-frequency
    route_df["ROUTE_TYPE"] = "Standard"

    fob_75 = route_df["TOTAL_FOB_INR"].quantile(0.75)
    ship_75 = route_df["NUM_SHIPMENTS"].quantile(0.75)

    route_df.loc[route_df["TOTAL_FOB_INR"] > fob_75, "ROUTE_TYPE"] = "High-Value"
    route_df.loc[route_df["NUM_SHIPMENTS"] > ship_75, "ROUTE_TYPE"] = "High-Frequency"

    # Both high = Premium
    high_val = route_df["TOTAL_FOB_INR"] > fob_75
    high_freq = route_df["NUM_SHIPMENTS"] > ship_75
    route_df.loc[high_val & high_freq, "ROUTE_TYPE"] = "Premium"

    # Sort by FOB value
    route_df = route_df.sort_values("TOTAL_FOB_INR", ascending=False)

    return route_df


def analyze_transport_modes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze transport mode distribution.
    Shows which shipping modes are used most and their FOB value.
    """

    if "MODE_OF_PORT" not in df.columns or "FOB_INR" not in df.columns:
        return pd.DataFrame()

    mode_analysis = df.groupby("MODE_OF_PORT").agg({
        "FOB_INR": ["sum", "mean", "count"],
        "QUANTITY": "sum",
        "SB_NO": "nunique",
        "COUNTRY": "nunique"
    }).reset_index()

    mode_analysis.columns = ["MODE_OF_TRANSPORT", "TOTAL_FOB_INR", "AVG_FOB_PER_SHIPMENT",
                             "NUM_SHIPMENTS", "TOTAL_QUANTITY", "UNIQUE_SHIPMENTS",
                             "NUM_COUNTRIES"]

    # Calculate percentage of total
    total_fob = mode_analysis["TOTAL_FOB_INR"].sum()
    mode_analysis["PERCENTAGE_OF_TOTAL_FOB"] = (
        (mode_analysis["TOTAL_FOB_INR"] / total_fob) * 100
    ).round(2)

    mode_analysis = mode_analysis.sort_values("TOTAL_FOB_INR", ascending=False)

    return mode_analysis


# ============================================================
# UNIT & PRODUCT TRENDS
# ============================================================

def analyze_product_trends(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze product descriptions by value and quantity.
    Shows top products, market concentration, and distribution.
    """

    log("Analyzing product trends", "INFO")

    if "PRODUCT_DESCRIPTION" not in df.columns:
        return pd.DataFrame()

    product_analysis = df.groupby("PRODUCT_DESCRIPTION").agg({
        "FOB_INR": ["sum", "mean", "count"],
        "QUANTITY": ["sum", "mean"],
        "UNIT": lambda x: x.mode()[0] if len(x.mode()) > 0 else "Unknown",
        "IMPORTER_NAME": "nunique",
        "COUNTRY": "nunique"
    }).reset_index()

    product_analysis.columns = ["PRODUCT_DESCRIPTION", "TOTAL_FOB_INR", "AVG_FOB_PER_SHIPMENT",
                                "NUM_SHIPMENTS", "TOTAL_QUANTITY", "AVG_QUANTITY_PER_SHIPMENT",
                                "PRIMARY_UNIT", "NUM_IMPORTERS", "NUM_COUNTRIES"]

    # Calculate percentage of total FOB
    total_fob = product_analysis["TOTAL_FOB_INR"].sum()
    product_analysis["PERCENTAGE_OF_TOTAL_FOB"] = (
        (product_analysis["TOTAL_FOB_INR"] / total_fob) * 100
    ).round(2)

    # Cumulative for Pareto analysis
    product_analysis = product_analysis.sort_values("TOTAL_FOB_INR", ascending=False)
    product_analysis["CUMULATIVE_PERCENTAGE"] = product_analysis["PERCENTAGE_OF_TOTAL_FOB"].cumsum().round(2)

    return product_analysis


def analyze_unit_distribution(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze unit types distribution.
    Shows what units are used, their FOB values, and quantities.
    """

    if "UNIT" not in df.columns:
        return pd.DataFrame()

    unit_analysis = df.groupby("UNIT").agg({
        "FOB_INR": ["sum", "mean", "count"],
        "QUANTITY": "sum",
        "PRODUCT_DESCRIPTION": lambda x: x.mode()[0] if len(x.mode()) > 0 else "Unknown",
        "IMPORTER_NAME": "nunique"
    }).reset_index()

    unit_analysis.columns = ["UNIT", "TOTAL_FOB_INR", "AVG_FOB_PER_SHIPMENT",
                             "NUM_SHIPMENTS", "TOTAL_QUANTITY", "MOST_COMMON_PRODUCT",
                             "NUM_IMPORTERS"]

    # Percentage
    total_fob = unit_analysis["TOTAL_FOB_INR"].sum()
    unit_analysis["PERCENTAGE_OF_TOTAL_FOB"] = (
        (unit_analysis["TOTAL_FOB_INR"] / total_fob) * 100
    ).round(2)

    unit_analysis = unit_analysis.sort_values("TOTAL_FOB_INR", ascending=False)

    return unit_analysis


# ============================================================
# PRICE VARIABILITY & TRENDS
# ============================================================

def analyze_price_variability(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze price variability per importer.
    Shows which importers have stable vs volatile pricing.
    Helps understand pricing negotiations & consistency.
    """

    log("Analyzing price variability per importer", "INFO")

    if "IMPORTER_NAME" not in df.columns or "UNIT_RATE_INR" not in df.columns:
        return pd.DataFrame()

    importer_price = df.groupby("IMPORTER_NAME").agg({
        "UNIT_RATE_INR": ["mean", "std", "min", "max", "count"],
        "FOB_INR": ["mean", "std", "min", "max"]
    }).reset_index()

    importer_price.columns = ["IMPORTER_NAME", "AVG_UNIT_PRICE_INR", "UNIT_PRICE_STD",
                              "MIN_UNIT_PRICE_INR", "MAX_UNIT_PRICE_INR", "PRICE_DATA_POINTS",
                              "AVG_FOB_INR", "FOB_STD", "MIN_FOB_INR", "MAX_FOB_INR"]

    # Calculate coefficient of variation (CV) for price volatility
    # CV = (Std Dev / Mean) * 100
    importer_price["UNIT_PRICE_VOLATILITY_%"] = (
        (importer_price["UNIT_PRICE_STD"] / importer_price["AVG_UNIT_PRICE_INR"] * 100)
    ).round(2)

    importer_price["FOB_VOLATILITY_%"] = (
        (importer_price["FOB_STD"] / importer_price["AVG_FOB_INR"] * 100)
    ).round(2)

    # Classification of price stability
    importer_price["PRICE_STABILITY"] = "Volatile"  # Default to volatile
    importer_price.loc[importer_price["UNIT_PRICE_VOLATILITY_%"] <= 15, "PRICE_STABILITY"] = "Very Stable"
    importer_price.loc[
        (importer_price["UNIT_PRICE_VOLATILITY_%"] > 15) &
        (importer_price["UNIT_PRICE_VOLATILITY_%"] <= 30),
        "PRICE_STABILITY"
    ] = "Stable"

    # Remove NaN rows (where std is 0 or only 1 data point)
    importer_price = importer_price.dropna(subset=["UNIT_PRICE_VOLATILITY_%"])

    importer_price = importer_price.sort_values("UNIT_PRICE_VOLATILITY_%", ascending=False)

    return importer_price


def analyze_country_price_trends(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze price variability by country.
    Shows which countries have volatile vs stable pricing.
    """

    if "COUNTRY" not in df.columns or "FOB_INR" not in df.columns:
        return pd.DataFrame()

    country_price = df.groupby("COUNTRY").agg({
        "FOB_INR": ["mean", "std", "min", "max", "count"],
        "UNIT_RATE_INR": ["mean", "std"],
        "IMPORTER_NAME": "nunique"
    }).reset_index()

    country_price.columns = ["COUNTRY", "AVG_FOB_INR", "FOB_STD", "MIN_FOB_INR",
                             "MAX_FOB_INR", "NUM_SHIPMENTS", "AVG_UNIT_PRICE",
                             "UNIT_PRICE_STD", "NUM_IMPORTERS"]

    country_price["FOB_VOLATILITY_%"] = (
        (country_price["FOB_STD"] / country_price["AVG_FOB_INR"] * 100)
    ).round(2)

    country_price["UNIT_PRICE_VOLATILITY_%"] = (
        (country_price["UNIT_PRICE_STD"] / country_price["AVG_UNIT_PRICE"] * 100)
    ).round(2)

    country_price = country_price.dropna(subset=["FOB_VOLATILITY_%"])
    country_price = country_price.sort_values("FOB_VOLATILITY_%", ascending=False)

    return country_price


# ============================================================
# MARKET CONCENTRATION & IMPORTER SHARE
# ============================================================

def analyze_market_concentration(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze market concentration - percentage of FOB per importer.
    Useful for understanding market dominance and fragmentation.
    Pareto principle: 80/20 rule - do 20% of importers drive 80% of value?
    """

    log("Analyzing market concentration", "INFO")

    if "IMPORTER_NAME" not in df.columns or "FOB_INR" not in df.columns:
        return pd.DataFrame()

    importer_share = df.groupby("IMPORTER_NAME").agg({
        "FOB_INR": "sum",
        "SB_NO": "nunique",
        "QUANTITY": "sum",
        "PRODUCT_DESCRIPTION": "nunique",
        "COUNTRY": "nunique"
    }).reset_index()

    importer_share.columns = ["IMPORTER_NAME", "TOTAL_FOB_INR", "NUM_SHIPMENTS",
                              "TOTAL_QUANTITY", "NUM_UNIQUE_PRODUCTS", "NUM_COUNTRIES"]

    # Calculate percentage of total FOB
    total_fob = importer_share["TOTAL_FOB_INR"].sum()
    importer_share["PERCENTAGE_OF_TOTAL_FOB"] = (
        (importer_share["TOTAL_FOB_INR"] / total_fob) * 100
    ).round(2)

    # Cumulative percentage (for Pareto analysis)
    importer_share = importer_share.sort_values("TOTAL_FOB_INR", ascending=False)
    importer_share["CUMULATIVE_PERCENTAGE"] = importer_share["PERCENTAGE_OF_TOTAL_FOB"].cumsum().round(2)

    # Market concentration classification
    importer_share["MARKET_CATEGORY"] = "Micro"
    importer_share.loc[importer_share["PERCENTAGE_OF_TOTAL_FOB"] >= 0.5, "MARKET_CATEGORY"] = "Small"
    importer_share.loc[importer_share["PERCENTAGE_OF_TOTAL_FOB"] >= 2, "MARKET_CATEGORY"] = "Medium"
    importer_share.loc[importer_share["PERCENTAGE_OF_TOTAL_FOB"] >= 5, "MARKET_CATEGORY"] = "Large"
    importer_share.loc[importer_share["PERCENTAGE_OF_TOTAL_FOB"] >= 10, "MARKET_CATEGORY"] = "Major"

    return importer_share


def analyze_country_concentration(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze market concentration by country.
    Shows market fragmentation by destination.
    """

    if "COUNTRY" not in df.columns or "FOB_INR" not in df.columns:
        return pd.DataFrame()

    country_share = df.groupby("COUNTRY").agg({
        "FOB_INR": "sum",
        "SB_NO": "nunique",
        "IMPORTER_NAME": "nunique",
        "PRODUCT_DESCRIPTION": "nunique"
    }).reset_index()

    country_share.columns = ["COUNTRY", "TOTAL_FOB_INR", "NUM_SHIPMENTS",
                             "NUM_IMPORTERS", "NUM_UNIQUE_PRODUCTS"]

    total_fob = country_share["TOTAL_FOB_INR"].sum()
    country_share["PERCENTAGE_OF_TOTAL_FOB"] = (
        (country_share["TOTAL_FOB_INR"] / total_fob) * 100
    ).round(2)

    # Concentration ratio: number of importers per country
    # Higher = more fragmented; Lower = concentrated
    country_share["MARKET_FRAGMENTATION_INDEX"] = (
        country_share["NUM_IMPORTERS"] / country_share["NUM_SHIPMENTS"]
    ).round(2)

    country_share = country_share.sort_values("TOTAL_FOB_INR", ascending=False)

    return country_share


# ============================================================
# PRODUCT & UNIT CONCENTRATION
# ============================================================

def analyze_product_concentration(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze product concentration in the market.
    Shows which products drive the market.
    """

    if "PRODUCT_DESCRIPTION" not in df.columns or "FOB_INR" not in df.columns:
        return pd.DataFrame()

    product_conc = df.groupby("PRODUCT_DESCRIPTION").agg({
        "FOB_INR": "sum",
        "SB_NO": "nunique",
        "IMPORTER_NAME": "nunique"
    }).reset_index()

    product_conc.columns = ["PRODUCT_DESCRIPTION", "TOTAL_FOB_INR", "NUM_SHIPMENTS", "NUM_IMPORTERS"]

    total_fob = product_conc["TOTAL_FOB_INR"].sum()
    product_conc["PERCENTAGE_OF_TOTAL_FOB"] = (
        (product_conc["TOTAL_FOB_INR"] / total_fob) * 100
    ).round(2)

    product_conc = product_conc.sort_values("TOTAL_FOB_INR", ascending=False)
    product_conc["CUMULATIVE_PERCENTAGE"] = product_conc["PERCENTAGE_OF_TOTAL_FOB"].cumsum().round(2)

    return product_conc


# ============================================================
# EXCEL EXPORT WITH HS CODES
# ============================================================

def export_to_excel(
    cleaned_df: pd.DataFrame,
    intelligence_df: pd.DataFrame,
    gme_df: pd.DataFrame,
    gulf_df: pd.DataFrame,
    me_df: pd.DataFrame,
    by_country_df: pd.DataFrame,
    routes_df: pd.DataFrame = None,
    modes_df: pd.DataFrame = None,
    products_df: pd.DataFrame = None,
    units_df: pd.DataFrame = None,
    importer_volatility: pd.DataFrame = None,
    country_volatility: pd.DataFrame = None,
    importer_conc: pd.DataFrame = None,
    country_conc: pd.DataFrame = None,
    product_conc: pd.DataFrame = None
):
    """Export all analysis to Excel with HS codes and advanced analytics."""

    log("Exporting to Excel workbook with HS codes & advanced analytics", "INFO")

    filename = "HS6815_Importer_Intelligence.xlsx"

    with pd.ExcelWriter(filename, engine='openpyxl') as writer:

        summary_data = {
            "Metric": ["Total Transactions", "Total FOB Value (INR)", "Unique Importers", "Unique HS Codes", "Unique Countries", "Gulf & ME Importers", "Long-term Importers (2+ years)", "Export Date"],
            "Value": [len(cleaned_df), cleaned_df["FOB_INR"].sum() if "FOB_INR" in cleaned_df.columns else 0, cleaned_df["IMPORTER_NAME"].nunique(dropna=True), cleaned_df["HS_CODE"].nunique(dropna=True) if "HS_CODE" in cleaned_df.columns else 0, cleaned_df["COUNTRY"].nunique(dropna=True) if "COUNTRY" in cleaned_df.columns else 0, len(gme_df), (intelligence_df["IS_LONG_TERM"] == "Yes").sum() if "IS_LONG_TERM" in intelligence_df.columns else 0, datetime.now().strftime("%Y-%m-%d %H:%M:%S")]
        }
        summary_df = pd.DataFrame(summary_data)
        summary_df.to_excel(writer, sheet_name="Executive Summary", index=False)

        intelligence_df.to_excel(writer, sheet_name="Importer Intelligence", index=False)

        gulf_importers = intelligence_df[intelligence_df["IMPORTER_NAME"].isin(gulf_df["IMPORTER_NAME"].unique())].copy()
        gulf_importers.to_excel(writer, sheet_name="Gulf Importers", index=False)

        me_importers = intelligence_df[intelligence_df["IMPORTER_NAME"].isin(me_df["IMPORTER_NAME"].unique())].copy()
        me_importers.to_excel(writer, sheet_name="Middle East Importers", index=False)

        by_country_df.to_excel(writer, sheet_name="By Country", index=False)

        export_cols = ["IMPORTER_NAME", "IMPORTER_ADDRESS", "EXPORTER_NAME", "COUNTRY", "HS_CODE", "HS_2", "HS_4", "HS_6", "PRODUCT_DESCRIPTION", "QUANTITY", "UNIT", "FOB_INR", "UNIT_RATE_INR", "SB_DATE", "SB_NO", "INVOICE_NO", "SOURCE_FILE"]
        export_cols = [c for c in export_cols if c in cleaned_df.columns]
        cleaned_df[export_cols].to_excel(writer, sheet_name="Full Transactions", index=False)

        if routes_df is not None and not routes_df.empty:
            routes_df.to_excel(writer, sheet_name="Routes Analysis", index=False)
            log("  Sheet 7: Routes Analysis ✓")

        if modes_df is not None and not modes_df.empty:
            modes_df.to_excel(writer, sheet_name="Transport Modes", index=False)
            log("  Sheet 8: Transport Modes ✓")

        if products_df is not None and not products_df.empty:
            products_df.to_excel(writer, sheet_name="Product Trends", index=False)
            log("  Sheet 9: Product Trends ✓")

        if units_df is not None and not units_df.empty:
            units_df.to_excel(writer, sheet_name="Unit Distribution", index=False)
            log("  Sheet 10: Unit Distribution ✓")

        if importer_volatility is not None and not importer_volatility.empty:
            importer_volatility.to_excel(writer, sheet_name="Price Volatility-Importers", index=False)
            log("  Sheet 11: Price Volatility-Importers ✓")

        if country_volatility is not None and not country_volatility.empty:
            country_volatility.to_excel(writer, sheet_name="Price Volatility-Countries", index=False)
            log("  Sheet 12: Price Volatility-Countries ✓")

        if importer_conc is not None and not importer_conc.empty:
            importer_conc.to_excel(writer, sheet_name="Market Share-Importers", index=False)
            log("  Sheet 13: Market Share-Importers ✓")

        if country_conc is not None and not country_conc.empty:
            country_conc.to_excel(writer, sheet_name="Market Share-Countries", index=False)
            log("  Sheet 14: Market Share-Countries ✓")

        if product_conc is not None and not product_conc.empty:
            product_conc.to_excel(writer, sheet_name="Product Concentration", index=False)
            log("  Sheet 15: Product Concentration ✓")

    log(f"Excel workbook created: {filename}", "SUCCESS")
    return filename


# ============================================================
# MAIN PIPELINE
# ============================================================

def main():
    """Execute complete pipeline."""

    print("\n" + "=" * 70)
    print("HS 6815 GULF & MIDDLE EAST IMPORTER INTELLIGENCE SYSTEM")
    print("FINAL VERSION - With HS Codes & Enhanced Importer Cleaning")
    print("=" * 70 + "\n")

    # STEP 1
    log("STEP 1: File Upload", "INFO")
    uploaded, valid_files = upload_files()

    # STEP 2
    log("\nSTEP 2: Ingestion & Mapping", "INFO")
    raw_df = ingest_all_files(uploaded, valid_files)

    # STEP 4
    log("\nSTEP 4: Error Replacement", "INFO")
    raw_df = step_4_error_replacement(raw_df)

    # STEP 4.2
    log("\nSTEP 4.2: Column Cleaning (Enhanced Importer + HS Codes)", "INFO")
    cleaned_df = clean_and_prepare(raw_df)

    # STEP 4.3
    log("\nSTEP 4.3: Unit Price Calculation", "INFO")
    cleaned_df = calculate_unit_prices(cleaned_df)

    # STEP 5
    log("\nSTEP 5: Deduplication", "INFO")
    cleaned_df, dedup_stats = deduplicate_smart(cleaned_df)
    log(f"Removed {dedup_stats['removed']} duplicates", "SUCCESS")

    # STEP 6
    log("\nSTEP 6: Importer Intelligence Calculation", "INFO")
    intelligence_df = calculate_importer_intelligence(cleaned_df)

    # STEP 7
    log("\nSTEP 7: Regional Analysis", "INFO")
    gme_df, gulf_df, me_df = analyze_gulf_middle_east(cleaned_df)

    # Port & Route Analysis
    log("\nSTEP 7.2: Port & Route Analysis", "INFO")
    routes_df = analyze_port_and_routes(cleaned_df)
    modes_df = analyze_transport_modes(cleaned_df)

    # Product & Unit Trends
    log("\nSTEP 7.3: Product & Unit Trends", "INFO")
    products_df = analyze_product_trends(cleaned_df)
    units_df = analyze_unit_distribution(cleaned_df)

    # Price Analysis
    log("\nSTEP 7.4: Price Variability Analysis", "INFO")
    importer_volatility = analyze_price_variability(cleaned_df)
    country_volatility = analyze_country_price_trends(cleaned_df)

    # Market Concentration
    log("\nSTEP 7.5: Market Concentration Analysis", "INFO")
    importer_conc = analyze_market_concentration(cleaned_df)
    country_conc = analyze_country_concentration(cleaned_df)
    product_conc = analyze_product_concentration(cleaned_df)

    print(f"\n🌍 Gulf & Middle East breakdown:")
    print(f"  Total rows: {len(cleaned_df)}")
    print(f"  Gulf & ME: {len(gme_df)} ({len(gme_df)/len(cleaned_df)*100:.1f}%)")
    print(f"  Gulf only: {len(gulf_df)} ({len(gulf_df)/len(cleaned_df)*100:.1f}%)")
    print(f"  Middle East only: {len(me_df)} ({len(me_df)/len(cleaned_df)*100:.1f}%)")

    # By country
    by_country_df = analyze_by_country(cleaned_df)

    print(f"\n📊 Unique HS Codes found: {cleaned_df['HS_CODE'].nunique(dropna=True)}")

    # STEP 8
    log("\nSTEP 8: Export to Excel", "INFO")
    excel_file = export_to_excel(cleaned_df, intelligence_df, gme_df, gulf_df, me_df, by_country_df, routes_df, modes_df, products_df, units_df, importer_volatility,
                                 country_volatility, importer_conc, country_conc, product_conc)

    # Download
    if IN_COLAB:
        files.download(excel_file)
        log(f"File ready for download: {excel_file}", "SUCCESS")

    print("\n" + "=" * 70)
    log("PIPELINE COMPLETED SUCCESSFULLY", "SUCCESS")
    print("=" * 70 + "\n")

    return {
        "cleaned_df": cleaned_df,
        "intelligence_df": intelligence_df,
        "gme_df": gme_df,
        "by_country_df": by_country_df,
        "excel_file": excel_file
    }


# ============================================================
# RUN PIPELINE
# ============================================================

if __name__ == "__main__":
    results = main()


STEP 0: IMPORTS & CONFIGURATION
✅ pandas imported
✅ numpy imported
✅ openpyxl imported
✅ Google Colab detected
✅ Pandas configured
✅ Configuration: HS 6815
✅ Mappings loaded
✅ Column map configured

✅ STEP 0 COMPLETED


HS 6815 GULF & MIDDLE EAST IMPORTER INTELLIGENCE SYSTEM
FINAL VERSION - With HS Codes & Enhanced Importer Cleaning

ℹ️  STEP 1: File Upload
ℹ️  Please upload monthly import files (XLS / XLSX / CSV).


Saving 68 exp Feb2025.xlsx to 68 exp Feb2025 (1).xlsx
Saving 68 exp JUL25.xlsx to 68 exp JUL25 (1).xlsx
Saving 68 exp Mar 2025.xlsx to 68 exp Mar 2025 (1).xlsx
Saving 68 Export June 25.xlsx to 68 Export June 25 (1).xlsx
Saving April23EXP2.xlsx_68.xlsx to April23EXP2.xlsx_68 (1).xlsx
Saving April24_68.xlsx to April24_68 (1).xlsx
Saving Aug23EXP1.xlsx_68.xlsx to Aug23EXP1.xlsx_68 (1).xlsx
Saving Aug24EXP2.xlsx_68.xlsx to Aug24EXP2.xlsx_68 (1).xlsx
Saving ch 68 exp feb 24.xlsx to ch 68 exp feb 24 (1).xlsx
Saving ch 68 exp sep24.xlsx to ch 68 exp sep24 (1).xlsx
Saving Chapter 68 Dec 22.xlsx to Chapter 68 Dec 22 (1).xlsx
Saving Chapter 68 July 22.xlsx to Chapter 68 July 22 (1).xlsx
Saving Chapter 68 Jun 22.xlsx to Chapter 68 Jun 22 (1).xlsx
Saving Chapter 68 May 22.xlsx to Chapter 68 May 22.xlsx
Saving Chapter 68 Nov 22.xlsx to Chapter 68 Nov 22 (1).xlsx
Saving Chapter 68 Sep 22.xlsx to Chapter 68 Sep 22 (1).xlsx
Saving Dec23EXP2.xlsx_68.xlsx to Dec23EXP2.xlsx_68 (1).xlsx
Saving EXP68DEC24.

/tmp/ipython-input-2812132873.py:574: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["SB_DATE"] = pd.to_datetime(df["SB_DATE"], errors="coerce", infer_datetime_format=True)


ℹ️  
STEP 4.3: Unit Price Calculation
ℹ️  Calculating unit prices (with fallback logic)
ℹ️  
STEP 5: Deduplication
✅ Removed 83373 duplicates
ℹ️  
STEP 6: Importer Intelligence Calculation
ℹ️  Calculating importer intelligence metrics
✅ Intelligence calculated for 5801 importers
ℹ️  
STEP 7: Regional Analysis
ℹ️  
STEP 7.2: Port & Route Analysis
ℹ️  Analyzing ports & routes
ℹ️  
STEP 7.3: Product & Unit Trends
ℹ️  Analyzing product trends
ℹ️  
STEP 7.4: Price Variability Analysis
ℹ️  Analyzing price variability per importer
ℹ️  
STEP 7.5: Market Concentration Analysis
ℹ️  Analyzing market concentration

🌍 Gulf & Middle East breakdown:
  Total rows: 12596
  Gulf & ME: 875 (6.9%)
  Gulf only: 663 (5.3%)
  Middle East only: 212 (1.7%)

📊 Unique HS Codes found: 10
ℹ️  
STEP 8: Export to Excel
ℹ️  Exporting to Excel workbook with HS codes & advanced analytics
ℹ️    Sheet 7: Routes Analysis ✓
ℹ️    Sheet 9: Product Trends ✓
ℹ️    Sheet 10: Unit Distribution ✓
ℹ️    Sheet 11: Price Volatility

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ File ready for download: HS6815_Importer_Intelligence.xlsx

✅ PIPELINE COMPLETED SUCCESSFULLY



In [ ]:
pip install dash dash_bootstrap_components

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 14.9 MB/s eta 0:00:00


In [ ]:
"""
Gulf & Middle East Importer Prospecting Dashboard
Interactive dashboard for April 2026 visit planning
"""

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, dash_table, Input, Output, State
import dash_bootstrap_components as dbc
import numpy as np

# Load all data sheets
print("📊 Loading data...")
file_path = '/content/HS6815_Importer_Intelligence (2).xlsx'

# Load datasets
exec_summary = pd.read_excel(file_path, sheet_name='Executive Summary')
importer_intel = pd.read_excel(file_path, sheet_name='Importer Intelligence')
gulf_importers = pd.read_excel(file_path, sheet_name='Gulf Importers')
me_importers = pd.read_excel(file_path, sheet_name='Middle East Importers')
by_country = pd.read_excel(file_path, sheet_name='By Country')
full_transactions = pd.read_excel(file_path, sheet_name='Full Transactions')
routes = pd.read_excel(file_path, sheet_name='Routes Analysis')
product_trends = pd.read_excel(file_path, sheet_name='Product Trends')
price_vol_importers = pd.read_excel(file_path, sheet_name='Price Volatility-Importers')
price_vol_countries = pd.read_excel(file_path, sheet_name='Price Volatility-Countries')
market_share_importers = pd.read_excel(file_path, sheet_name='Market Share-Importers')
market_share_countries = pd.read_excel(file_path, sheet_name='Market Share-Countries')

# Define Gulf and Middle East countries
GULF_COUNTRIES = ['UNITED ARAB EMIRATES', 'SAUDI ARABIA', 'QATAR', 'KUWAIT', 'OMAN', 'BAHRAIN']
ME_COUNTRIES = ['IRAN', 'IRAQ', 'JORDAN', 'ISRAEL', 'TURKEY', 'EGYPT', 'LEBANON', 'YEMEN']

# Combine for filtering
gulf_me_countries = GULF_COUNTRIES + ME_COUNTRIES

# Filter country data to Gulf & ME only
gulf_me_country_data = by_country[by_country['COUNTRY'].isin(gulf_me_countries)].copy()

# Add region classification
def classify_region(country):
    if country in GULF_COUNTRIES:
        return 'Gulf'
    elif country in ME_COUNTRIES:
        return 'Middle East'
    else:
        return 'Other'

gulf_me_country_data['REGION'] = gulf_me_country_data['COUNTRY'].apply(classify_region)

# Combine Gulf and ME importers
gulf_me_importers = pd.concat([gulf_importers, me_importers], ignore_index=True)

# Get transactions for Gulf & ME only
gulf_me_transactions = full_transactions[full_transactions['COUNTRY'].isin(gulf_me_countries)].copy()
gulf_me_transactions['REGION'] = gulf_me_transactions['COUNTRY'].apply(classify_region)

print(f"✅ Data loaded: {len(gulf_me_importers)} Gulf & ME importers, {len(gulf_me_transactions)} transactions")

# Initialize Dash app with Bootstrap theme
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Define color scheme
COLORS = {
    'gulf': '#1f77b4',      # Blue
    'me': '#ff7f0e',        # Orange
    'background': '#f8f9fa',
    'text': '#212529',
    'accent': '#28a745',    # Green
    'warning': '#ffc107',   # Yellow
    'danger': '#dc3545'     # Red
}

# App layout
app.layout = dbc.Container([
    # Header
    dbc.Row([
        dbc.Col([
            html.H1("🌍 Gulf & Middle East Prospecting Dashboard",
                   className="text-center mb-1 mt-3",
                   style={'color': COLORS['text'], 'fontWeight': 'bold'}),
            html.H5("April 2026 Visit Planning - Importer Intelligence",
                   className="text-center text-muted mb-4")
        ])
    ]),

    # Executive Summary Cards
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    html.H6("Total Importers", className="text-muted"),
                    html.H3(f"{len(gulf_me_importers):,}", className="text-primary")
                ])
            ])
        ], width=2),
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    html.H6("Total FOB Value", className="text-muted"),
                    html.H3(f"₹{gulf_me_importers['TOTAL_FOB_INR'].sum()/1e7:.1f}Cr", className="text-success")
                ])
            ])
        ], width=2),
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    html.H6("Countries", className="text-muted"),
                    html.H3(f"{len(gulf_me_country_data)}", className="text-info")
                ])
            ])
        ], width=2),
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    html.H6("With Address", className="text-muted"),
                    html.H3(f"{gulf_me_importers[gulf_me_importers['HAS_ADDRESS']=='Yes'].shape[0]:,}",
                           className="text-warning")
                ])
            ])
        ], width=2),
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    html.H6("Long-term (2+ yrs)", className="text-muted"),
                    html.H3(f"{gulf_me_importers[gulf_me_importers['IS_LONG_TERM']=='Yes'].shape[0]:,}",
                           className="text-danger")
                ])
            ])
        ], width=2),
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    html.H6("Avg Deal Size", className="text-muted"),
                    html.H3(f"₹{gulf_me_importers['AVG_TRANSACTION_VALUE'].mean()/1e5:.1f}L",
                           className="text-secondary")
                ])
            ])
        ], width=2),
    ], className="mb-4"),

    # Global Filters
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    html.H5("🎯 Filters", className="mb-3"),

                    html.Label("Region:", className="fw-bold"),
                    dcc.Dropdown(
                        id='region-filter',
                        options=[
                            {'label': '🌐 All Gulf & Middle East', 'value': 'ALL'},
                            {'label': '🏙️ Gulf Countries Only', 'value': 'GULF'},
                            {'label': '🗺️ Middle East Only', 'value': 'ME'}
                        ],
                        value='ALL',
                        className="mb-3"
                    ),

                    html.Label("Countries:", className="fw-bold"),
                    dcc.Dropdown(
                        id='country-filter',
                        options=[{'label': c, 'value': c} for c in sorted(gulf_me_countries)],
                        value=gulf_me_countries,
                        multi=True,
                        className="mb-3"
                    ),

                    html.Label("Has Address:", className="fw-bold"),
                    dcc.Checklist(
                        id='address-filter',
                        options=[{'label': ' Only importers with verified address', 'value': 'YES'}],
                        value=['YES'],
                        className="mb-3"
                    ),

                    html.Label("Long-term Importers (2+ years):", className="fw-bold"),
                    dcc.Checklist(
                        id='longterm-filter',
                        options=[{'label': ' Only established importers', 'value': 'YES'}],
                        value=[],
                        className="mb-3"
                    ),

                    html.Label("Minimum FOB Value (₹):", className="fw-bold"),
                    dcc.Slider(
                        id='fob-slider',
                        min=0,
                        max=int(gulf_me_importers['TOTAL_FOB_INR'].max()),
                        value=0,
                        marks={
                            0: '₹0',
                            int(gulf_me_importers['TOTAL_FOB_INR'].max()/4): f'₹{gulf_me_importers["TOTAL_FOB_INR"].max()/4/1e6:.0f}M',
                            int(gulf_me_importers['TOTAL_FOB_INR'].max()/2): f'₹{gulf_me_importers["TOTAL_FOB_INR"].max()/2/1e6:.0f}M',
                            int(gulf_me_importers['TOTAL_FOB_INR'].max()): f'₹{gulf_me_importers["TOTAL_FOB_INR"].max()/1e6:.0f}M'
                        },
                        tooltip={"placement": "bottom", "always_visible": True}
                    ),
                ])
            ])
        ], width=12)
    ], className="mb-4"),

    # Tabs for different views
    dbc.Tabs([
        # TAB 1: Country Overview
        dbc.Tab(label="📊 Country Overview", children=[
            dbc.Row([
                dbc.Col([
                    dcc.Graph(id='country-bar-chart')
                ], width=6),
                dbc.Col([
                    dcc.Graph(id='country-map')
                ], width=6)
            ], className="mt-3"),
            dbc.Row([
                dbc.Col([
                    dcc.Graph(id='country-metrics-table')
                ], width=12)
            ], className="mt-3")
        ]),

        # TAB 2: Top Importers
        dbc.Tab(label="🏢 Top Importers", children=[
            dbc.Row([
                dbc.Col([
                    dcc.Graph(id='importer-treemap')
                ], width=12)
            ], className="mt-3"),
            dbc.Row([
                dbc.Col([
                    html.H5("📋 Importer Details Table (Downloadable)", className="mt-3 mb-2"),
                    dbc.Button("📥 Download as Excel", id="download-button", color="success", className="mb-2"),
                    dcc.Download(id="download-excel"),
                    html.Div(id='importer-table')
                ], width=12)
            ])
        ]),

        # TAB 3: Product Analysis
        dbc.Tab(label="📦 Product Analysis", children=[
            dbc.Row([
                dbc.Col([
                    dcc.Graph(id='product-heatmap')
                ], width=12)
            ], className="mt-3"),
            dbc.Row([
                dbc.Col([
                    dcc.Graph(id='product-pareto')
                ], width=6),
                dbc.Col([
                    dcc.Graph(id='product-distribution')
                ], width=6)
            ], className="mt-3")
        ]),

        # TAB 4: Price Analysis
        dbc.Tab(label="💹 Price Analysis", children=[
            dbc.Row([
                dbc.Col([
                    dcc.Graph(id='price-box-plot')
                ], width=6),
                dbc.Col([
                    dcc.Graph(id='volatility-scatter')
                ], width=6)
            ], className="mt-3"),
            dbc.Row([
                dbc.Col([
                    dcc.Graph(id='price-comparison-bar')
                ], width=12)
            ], className="mt-3")
        ]),

        # TAB 5: Stability Quadrant
        dbc.Tab(label="⭐ Importer Stability", children=[
            dbc.Row([
                dbc.Col([
                    html.Div([
                        html.H5("🎯 Quadrant Analysis: Identify Best Prospects", className="mb-3"),
                        html.P([
                            html.Strong("Top Right (Star Partners): ", style={'color': COLORS['accent']}),
                            "High value + Loyal + Long-term → Priority meetings"
                        ]),
                        html.P([
                            html.Strong("Top Left (High-Value Switchers): ", style={'color': COLORS['warning']}),
                            "Big buyers but change vendors → Competitive pricing strategy"
                        ]),
                        html.P([
                            html.Strong("Bottom Right (Loyal Small): ", style={'color': '#17a2b8'}),
                            "Reliable but small orders → Quick courtesy meetings"
                        ]),
                        html.P([
                            html.Strong("Bottom Left (Risky): ", style={'color': COLORS['danger']}),
                            "Small + Volatile → Low priority"
                        ])
                    ], className="alert alert-info")
                ], width=12)
            ]),
            dbc.Row([
                dbc.Col([
                    dcc.Graph(id='stability-quadrant')
                ], width=12)
            ], className="mt-3")
        ]),

        # TAB 6: Meeting Planner
        dbc.Tab(label="📅 Meeting Planner", children=[
            dbc.Row([
                dbc.Col([
                    html.H5("🎯 Priority Meeting List", className="mt-3 mb-3"),
                    html.P("Importers ranked by strategic value (FOB × Loyalty × Long-term status)",
                          className="text-muted"),
                    html.Div(id='meeting-list')
                ], width=12)
            ])
        ])
    ], className="mt-3"),

    # Footer
    dbc.Row([
        dbc.Col([
            html.Hr(),
            html.P("💡 Dashboard created for Gulf & Middle East prospecting trip - April 2026",
                  className="text-center text-muted small")
        ])
    ], className="mt-5 mb-3")

], fluid=True, style={'backgroundColor': COLORS['background']})


# Callbacks for interactivity

# Combined filter callback
@app.callback(
    Output('country-filter', 'value'),
    Input('region-filter', 'value'),
    State('country-filter', 'value')
)
def update_country_filter(region, current_countries):
    if region == 'GULF':
        return GULF_COUNTRIES
    elif region == 'ME':
        return ME_COUNTRIES
    else:
        return gulf_me_countries


def filter_data(region, countries, has_address, is_longterm, min_fob):
    """Apply all filters to the importer dataset"""
    df = gulf_me_importers.copy()

    # Region filter
    if region == 'GULF':
        df = df[df['IMPORTER_NAME'].isin(gulf_importers['IMPORTER_NAME'])]
    elif region == 'ME':
        df = df[df['IMPORTER_NAME'].isin(me_importers['IMPORTER_NAME'])]

    # Country filter (via transactions)
    country_importers = gulf_me_transactions[gulf_me_transactions['COUNTRY'].isin(countries)]['IMPORTER_NAME'].unique()
    df = df[df['IMPORTER_NAME'].isin(country_importers)]

    # Address filter
    if 'YES' in has_address:
        df = df[df['HAS_ADDRESS'] == 'Yes']

    # Long-term filter
    if 'YES' in is_longterm:
        df = df[df['IS_LONG_TERM'] == 'Yes']

    # FOB minimum
    df = df[df['TOTAL_FOB_INR'] >= min_fob]

    return df


# Country bar chart
@app.callback(
    Output('country-bar-chart', 'figure'),
    [Input('region-filter', 'value'),
     Input('country-filter', 'value'),
     Input('address-filter', 'value'),
     Input('longterm-filter', 'value'),
     Input('fob-slider', 'value')]
)
def update_country_bar(region, countries, has_address, is_longterm, min_fob):
    filtered_df = filter_data(region, countries, has_address, is_longterm, min_fob)

    # Get country-level aggregation
    trans_filtered = gulf_me_transactions[gulf_me_transactions['IMPORTER_NAME'].isin(filtered_df['IMPORTER_NAME'])]
    country_agg = trans_filtered.groupby(['COUNTRY', 'REGION']).agg({
        'FOB_INR': 'sum',
        'IMPORTER_NAME': 'nunique',
        'SB_NO': 'count'
    }).reset_index()
    country_agg.columns = ['COUNTRY', 'REGION', 'TOTAL_FOB', 'NUM_IMPORTERS', 'NUM_SHIPMENTS']
    country_agg = country_agg.sort_values('TOTAL_FOB', ascending=True)

    fig = px.bar(country_agg,
                 y='COUNTRY',
                 x='TOTAL_FOB',
                 color='REGION',
                 color_discrete_map={'Gulf': COLORS['gulf'], 'Middle East': COLORS['me']},
                 text='NUM_IMPORTERS',
                 orientation='h',
                 title='Total Import Value by Country',
                 labels={'TOTAL_FOB': 'Total FOB Value (₹)', 'NUM_IMPORTERS': 'Importers'},
                 hover_data={'NUM_SHIPMENTS': True})

    fig.update_traces(texttemplate='%{text} importers', textposition='outside')
    fig.update_layout(height=500, showlegend=True)

    return fig


# Country map
@app.callback(
    Output('country-map', 'figure'),
    [Input('region-filter', 'value'),
     Input('country-filter', 'value'),
     Input('address-filter', 'value'),
     Input('longterm-filter', 'value'),
     Input('fob-slider', 'value')]
)
def update_country_map(region, countries, has_address, is_longterm, min_fob):
    filtered_df = filter_data(region, countries, has_address, is_longterm, min_fob)

    trans_filtered = gulf_me_transactions[gulf_me_transactions['IMPORTER_NAME'].isin(filtered_df['IMPORTER_NAME'])]
    country_agg = trans_filtered.groupby(['COUNTRY', 'REGION']).agg({
        'FOB_INR': 'sum',
        'IMPORTER_NAME': 'nunique'
    }).reset_index()

    # Create choropleth map
    fig = px.scatter_geo(country_agg,
                        locations='COUNTRY',
                        locationmode='country names',
                        size='FOB_INR',
                        color='REGION',
                        hover_name='COUNTRY',
                        hover_data={'FOB_INR': ':,.0f', 'IMPORTER_NAME': True},
                        color_discrete_map={'Gulf': COLORS['gulf'], 'Middle East': COLORS['me']},
                        title='Geographic Distribution of Importers',
                        projection='natural earth',
                        size_max=50)

    fig.update_geos(
        showcountries=True,
        countrycolor="lightgray",
        scope='asia',
        center=dict(lat=25, lon=50)
    )

    fig.update_layout(height=500)

    return fig


# Country metrics table
@app.callback(
    Output('country-metrics-table', 'figure'),
    [Input('region-filter', 'value'),
     Input('country-filter', 'value'),
     Input('address-filter', 'value'),
     Input('longterm-filter', 'value'),
     Input('fob-slider', 'value')]
)
def update_country_table(region, countries, has_address, is_longterm, min_fob):
    filtered_df = filter_data(region, countries, has_address, is_longterm, min_fob)

    trans_filtered = gulf_me_transactions[gulf_me_transactions['IMPORTER_NAME'].isin(filtered_df['IMPORTER_NAME'])]
    country_agg = trans_filtered.groupby('COUNTRY').agg({
        'FOB_INR': ['sum', 'mean'],
        'IMPORTER_NAME': 'nunique',
        'SB_NO': 'count',
        'HS_CODE': 'nunique'
    }).reset_index()

    country_agg.columns = ['Country', 'Total FOB (₹)', 'Avg FOB/Shipment', 'Importers', 'Shipments', 'HS Codes']
    country_agg = country_agg.sort_values('Total FOB (₹)', ascending=False)

    # Format currency
    country_agg['Total FOB (₹)'] = country_agg['Total FOB (₹)'].apply(lambda x: f'₹{x/1e6:.2f}M')
    country_agg['Avg FOB/Shipment'] = country_agg['Avg FOB/Shipment'].apply(lambda x: f'₹{x/1e5:.2f}L')

    fig = go.Figure(data=[go.Table(
        header=dict(values=list(country_agg.columns),
                   fill_color=COLORS['gulf'],
                   font=dict(color='white', size=12),
                   align='left'),
        cells=dict(values=[country_agg[col] for col in country_agg.columns],
                  fill_color='white',
                  align='left',
                  font=dict(size=11))
    )])

    fig.update_layout(title='Country-wise Detailed Metrics', height=400)

    return fig


# Importer treemap
@app.callback(
    Output('importer-treemap', 'figure'),
    [Input('region-filter', 'value'),
     Input('country-filter', 'value'),
     Input('address-filter', 'value'),
     Input('longterm-filter', 'value'),
     Input('fob-slider', 'value')]
)
def update_treemap(region, countries, has_address, is_longterm, min_fob):
    filtered_df = filter_data(region, countries, has_address, is_longterm, min_fob)

    # Get country for each importer
    importer_country = gulf_me_transactions.groupby('IMPORTER_NAME')['COUNTRY'].first().reset_index()
    filtered_with_country = filtered_df.merge(importer_country, on='IMPORTER_NAME')
    filtered_with_country = filtered_with_country[filtered_with_country['COUNTRY'].isin(countries)]

    # Top 50 by FOB
    top_importers = filtered_with_country.nlargest(50, 'TOTAL_FOB_INR')

    # Create loyalty color mapping
    top_importers['Loyalty_Category'] = pd.cut(top_importers['SUPPLIER_LOYALTY_SCORE'],
                                               bins=[0, 0.3, 0.6, 100],
                                               labels=['Low (Switcher)', 'Medium', 'High (Loyal)'])

    fig = px.treemap(top_importers,
                    path=['COUNTRY', 'IMPORTER_NAME'],
                    values='TOTAL_FOB_INR',
                    color='SUPPLIER_LOYALTY_SCORE',
                    color_continuous_scale='RdYlGn',
                    hover_data={'TOTAL_TRANSACTIONS': True,
                               'IS_LONG_TERM': True,
                               'HAS_ADDRESS': True,
                               'AVG_TRANSACTION_VALUE': ':,.0f'},
                    title='Top 50 Importers by Value (Size = FOB, Color = Loyalty Score)')

    fig.update_layout(height=600)

    return fig


# Importer table
@app.callback(
    Output('importer-table', 'children'),
    [Input('region-filter', 'value'),
     Input('country-filter', 'value'),
     Input('address-filter', 'value'),
     Input('longterm-filter', 'value'),
     Input('fob-slider', 'value')]
)
def update_importer_table(region, countries, has_address, is_longterm, min_fob):
    filtered_df = filter_data(region, countries, has_address, is_longterm, min_fob)

    # Get country for each importer
    importer_country = gulf_me_transactions.groupby('IMPORTER_NAME')['COUNTRY'].first().reset_index()
    filtered_with_country = filtered_df.merge(importer_country, on='IMPORTER_NAME')
    filtered_with_country = filtered_with_country[filtered_with_country['COUNTRY'].isin(countries)]

    # Select key columns for display
    display_cols = ['COUNTRY', 'IMPORTER_NAME', 'TOTAL_FOB_INR', 'TOTAL_TRANSACTIONS',
                   'AVG_TRANSACTION_VALUE', 'SUPPLIER_LOYALTY_SCORE', 'DAYS_ACTIVE',
                   'IS_LONG_TERM', 'HAS_ADDRESS', 'UNIQUE_HS_CODES', 'HS_CODES_LIST']

    table_df = filtered_with_country[display_cols].sort_values('TOTAL_FOB_INR', ascending=False).head(100)

    # Format numbers
    table_df['TOTAL_FOB_INR'] = table_df['TOTAL_FOB_INR'].apply(lambda x: f'₹{x/1e6:.2f}M')
    table_df['AVG_TRANSACTION_VALUE'] = table_df['AVG_TRANSACTION_VALUE'].apply(lambda x: f'₹{x/1e5:.2f}L')
    table_df['SUPPLIER_LOYALTY_SCORE'] = table_df['SUPPLIER_LOYALTY_SCORE'].round(2)

    # Rename columns for display
    table_df.columns = ['Country', 'Importer Name', 'Total FOB', 'Transactions',
                       'Avg Deal Size', 'Loyalty Score', 'Days Active',
                       'Long-term', 'Has Address', 'HS Codes', 'Products']

    return dash_table.DataTable(
        data=table_df.to_dict('records'),
        columns=[{'name': col, 'id': col} for col in table_df.columns],
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'padding': '8px',
            'fontSize': '11px',
            'fontFamily': 'Arial'
        },
        style_header={
            'backgroundColor': COLORS['gulf'],
            'color': 'white',
            'fontWeight': 'bold'
        },
        style_data_conditional=[
            {
                'if': {'filter_query': '{Long-term} = "Yes"'},
                'backgroundColor': '#d4edda',
            },
            {
                'if': {'filter_query': '{Has Address} = "Yes"'},
                'fontWeight': 'bold'
            }
        ],
        page_size=20,
        sort_action='native',
        filter_action='native'
    )


# Download Excel callback
@app.callback(
    Output("download-excel", "data"),
    Input("download-button", "n_clicks"),
    [State('region-filter', 'value'),
     State('country-filter', 'value'),
     State('address-filter', 'value'),
     State('longterm-filter', 'value'),
     State('fob-slider', 'value')],
    prevent_initial_call=True
)
def download_excel(n_clicks, region, countries, has_address, is_longterm, min_fob):
    filtered_df = filter_data(region, countries, has_address, is_longterm, min_fob)

    importer_country = gulf_me_transactions.groupby('IMPORTER_NAME')['COUNTRY'].first().reset_index()
    filtered_with_country = filtered_df.merge(importer_country, on='IMPORTER_NAME')
    filtered_with_country = filtered_with_country[filtered_with_country['COUNTRY'].isin(countries)]

    export_df = filtered_with_country.sort_values('TOTAL_FOB_INR', ascending=False)

    return dcc.send_data_frame(export_df.to_excel, "gulf_me_importers.xlsx", index=False)


# Product heatmap
@app.callback(
    Output('product-heatmap', 'figure'),
    [Input('region-filter', 'value'),
     Input('country-filter', 'value'),
     Input('address-filter', 'value'),
     Input('longterm-filter', 'value'),
     Input('fob-slider', 'value')]
)
def update_product_heatmap(region, countries, has_address, is_longterm, min_fob):
    filtered_df = filter_data(region, countries, has_address, is_longterm, min_fob)

    trans_filtered = gulf_me_transactions[
        (gulf_me_transactions['IMPORTER_NAME'].isin(filtered_df['IMPORTER_NAME'])) &
        (gulf_me_transactions['COUNTRY'].isin(countries))
    ]

    # Aggregate by country and product
    heatmap_data = trans_filtered.groupby(['COUNTRY', 'PRODUCT_DESCRIPTION']).agg({
        'FOB_INR': 'sum',
        'IMPORTER_NAME': 'nunique'
    }).reset_index()

    # Pivot for heatmap
    pivot_data = heatmap_data.pivot(index='COUNTRY', columns='PRODUCT_DESCRIPTION', values='FOB_INR')
    pivot_data = pivot_data.fillna(0)

    # Keep only top products
    top_products = heatmap_data.groupby('PRODUCT_DESCRIPTION')['FOB_INR'].sum().nlargest(10).index
    pivot_data = pivot_data[top_products]

    fig = px.imshow(pivot_data,
                   labels=dict(x="Product", y="Country", color="FOB Value (₹)"),
                   color_continuous_scale='Blues',
                   title='Product Demand Heatmap by Country (Top 10 Products)',
                   aspect='auto')

    fig.update_layout(height=500)

    return fig


# Product Pareto
@app.callback(
    Output('product-pareto', 'figure'),
    [Input('region-filter', 'value'),
     Input('country-filter', 'value'),
     Input('address-filter', 'value'),
     Input('longterm-filter', 'value'),
     Input('fob-slider', 'value')]
)
def update_pareto(region, countries, has_address, is_longterm, min_fob):
    filtered_df = filter_data(region, countries, has_address, is_longterm, min_fob)

    trans_filtered = gulf_me_transactions[
        (gulf_me_transactions['IMPORTER_NAME'].isin(filtered_df['IMPORTER_NAME'])) &
        (gulf_me_transactions['COUNTRY'].isin(countries))
    ]

    product_agg = trans_filtered.groupby('PRODUCT_DESCRIPTION')['FOB_INR'].sum().sort_values(ascending=False).reset_index()
    product_agg['Cumulative_%'] = (product_agg['FOB_INR'].cumsum() / product_agg['FOB_INR'].sum() * 100)
    product_agg = product_agg.head(15)

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(
        go.Bar(x=product_agg['PRODUCT_DESCRIPTION'], y=product_agg['FOB_INR'],
               name='FOB Value', marker_color=COLORS['gulf']),
        secondary_y=False
    )

    fig.add_trace(
        go.Scatter(x=product_agg['PRODUCT_DESCRIPTION'], y=product_agg['Cumulative_%'],
                  name='Cumulative %', mode='lines+markers', marker_color=COLORS['danger']),
        secondary_y=True
    )

    fig.update_xaxes(title_text="Product", tickangle=-45)
    fig.update_yaxes(title_text="FOB Value (₹)", secondary_y=False)
    fig.update_yaxes(title_text="Cumulative %", secondary_y=True)

    fig.update_layout(title='Product Pareto Analysis (80/20 Rule)', height=500)

    return fig


# Product distribution pie
@app.callback(
    Output('product-distribution', 'figure'),
    [Input('region-filter', 'value'),
     Input('country-filter', 'value'),
     Input('address-filter', 'value'),
     Input('longterm-filter', 'value'),
     Input('fob-slider', 'value')]
)
def update_product_pie(region, countries, has_address, is_longterm, min_fob):
    filtered_df = filter_data(region, countries, has_address, is_longterm, min_fob)

    trans_filtered = gulf_me_transactions[
        (gulf_me_transactions['IMPORTER_NAME'].isin(filtered_df['IMPORTER_NAME'])) &
        (gulf_me_transactions['COUNTRY'].isin(countries))
    ]

    product_agg = trans_filtered.groupby('PRODUCT_DESCRIPTION')['FOB_INR'].sum().sort_values(ascending=False).reset_index()
    top_10 = product_agg.head(10)

    fig = px.pie(top_10,
                values='FOB_INR',
                names='PRODUCT_DESCRIPTION',
                title='Top 10 Products by Value Share',
                hole=0.4)

    fig.update_layout(height=500)

    return fig


# Price box plot
@app.callback(
    Output('price-box-plot', 'figure'),
    [Input('region-filter', 'value'),
     Input('country-filter', 'value'),
     Input('address-filter', 'value'),
     Input('longterm-filter', 'value'),
     Input('fob-slider', 'value')]
)
def update_price_box(region, countries, has_address, is_longterm, min_fob):
    filtered_df = filter_data(region, countries, has_address, is_longterm, min_fob)

    trans_filtered = gulf_me_transactions[
        (gulf_me_transactions['IMPORTER_NAME'].isin(filtered_df['IMPORTER_NAME'])) &
        (gulf_me_transactions['COUNTRY'].isin(countries)) &
        (gulf_me_transactions['UNIT_RATE_INR'].notna()) &
        (gulf_me_transactions['UNIT_RATE_INR'] > 0)
    ]

    fig = px.box(trans_filtered,
                x='COUNTRY',
                y='UNIT_RATE_INR',
                color='REGION',
                color_discrete_map={'Gulf': COLORS['gulf'], 'Middle East': COLORS['me']},
                title='Unit Price Distribution by Country',
                labels={'UNIT_RATE_INR': 'Unit Rate (₹)'})

    fig.update_layout(height=500, showlegend=True)
    fig.update_xaxes(tickangle=-45)

    return fig


# Volatility scatter
@app.callback(
    Output('volatility-scatter', 'figure'),
    [Input('region-filter', 'value'),
     Input('country-filter', 'value'),
     Input('address-filter', 'value'),
     Input('longterm-filter', 'value'),
     Input('fob-slider', 'value')]
)
def update_volatility_scatter(region, countries, has_address, is_longterm, min_fob):
    # Filter price volatility data
    vol_filtered = price_vol_countries[price_vol_countries['COUNTRY'].isin(countries)].copy()
    vol_filtered['REGION'] = vol_filtered['COUNTRY'].apply(classify_region)

    fig = px.scatter(vol_filtered,
                    x='AVG_FOB_INR',
                    y='FOB_VOLATILITY_%',
                    size='NUM_SHIPMENTS',
                    color='REGION',
                    hover_name='COUNTRY',
                    color_discrete_map={'Gulf': COLORS['gulf'], 'Middle East': COLORS['me']},
                    title='Price Volatility vs Average Deal Size',
                    labels={'AVG_FOB_INR': 'Avg FOB per Shipment (₹)',
                           'FOB_VOLATILITY_%': 'Price Volatility (%)'},
                    size_max=40)

    # Add reference lines
    fig.add_hline(y=30, line_dash="dash", line_color="red",
                 annotation_text="High Volatility (>30%)")
    fig.add_hline(y=15, line_dash="dash", line_color="green",
                 annotation_text="Stable (<15%)")

    fig.update_layout(height=500)

    return fig


# Price comparison bar
@app.callback(
    Output('price-comparison-bar', 'figure'),
    [Input('region-filter', 'value'),
     Input('country-filter', 'value'),
     Input('address-filter', 'value'),
     Input('longterm-filter', 'value'),
     Input('fob-slider', 'value')]
)
def update_price_comparison(region, countries, has_address, is_longterm, min_fob):
    vol_filtered = price_vol_countries[price_vol_countries['COUNTRY'].isin(countries)].copy()
    vol_filtered['REGION'] = vol_filtered['COUNTRY'].apply(classify_region)
    vol_filtered = vol_filtered.sort_values('AVG_UNIT_PRICE', ascending=False)

    fig = go.Figure()

    # Add bars with error ranges
    for idx, row in vol_filtered.iterrows():
        fig.add_trace(go.Bar(
            x=[row['COUNTRY']],
            y=[row['AVG_UNIT_PRICE']],
            name=row['COUNTRY'],
            error_y=dict(
                type='data',
                symmetric=False,
                array=[row['AVG_UNIT_PRICE'] * row['UNIT_PRICE_VOLATILITY_%'] / 100],
                arrayminus=[row['AVG_UNIT_PRICE'] * row['UNIT_PRICE_VOLATILITY_%'] / 100]
            ),
            marker_color=COLORS['gulf'] if row['REGION'] == 'Gulf' else COLORS['me'],
            showlegend=False
        ))

    fig.update_layout(
        title='Average Unit Price by Country (with Volatility Range)',
        xaxis_title='Country',
        yaxis_title='Avg Unit Price (₹/unit)',
        height=500
    )
    fig.update_xaxes(tickangle=-45)

    return fig


# Stability quadrant
@app.callback(
    Output('stability-quadrant', 'figure'),
    [Input('region-filter', 'value'),
     Input('country-filter', 'value'),
     Input('address-filter', 'value'),
     Input('longterm-filter', 'value'),
     Input('fob-slider', 'value')]
)
def update_stability_quadrant(region, countries, has_address, is_longterm, min_fob):
    filtered_df = filter_data(region, countries, has_address, is_longterm, min_fob)

    # Get country for each importer
    importer_country = gulf_me_transactions.groupby('IMPORTER_NAME')['COUNTRY'].first().reset_index()
    filtered_with_country = filtered_df.merge(importer_country, on='IMPORTER_NAME')
    filtered_with_country = filtered_with_country[filtered_with_country['COUNTRY'].isin(countries)]

    # Calculate median values for quadrant lines
    median_loyalty = filtered_with_country['SUPPLIER_LOYALTY_SCORE'].median()
    median_fob = filtered_with_country['TOTAL_FOB_INR'].median()

    # Create quadrant labels
    def assign_quadrant(row):
        if row['SUPPLIER_LOYALTY_SCORE'] >= median_loyalty and row['TOTAL_FOB_INR'] >= median_fob:
            return '⭐ Star Partners'
        elif row['SUPPLIER_LOYALTY_SCORE'] < median_loyalty and row['TOTAL_FOB_INR'] >= median_fob:
            return '⚠️ High-Value Switchers'
        elif row['SUPPLIER_LOYALTY_SCORE'] >= median_loyalty and row['TOTAL_FOB_INR'] < median_fob:
            return '💚 Loyal Small Buyers'
        else:
            return '❌ Risky Prospects'

    filtered_with_country['Quadrant'] = filtered_with_country.apply(assign_quadrant, axis=1)

    fig = px.scatter(filtered_with_country,
                    x='SUPPLIER_LOYALTY_SCORE',
                    y='TOTAL_FOB_INR',
                    size='TOTAL_TRANSACTIONS',
                    color='Quadrant',
                    hover_name='IMPORTER_NAME',
                    hover_data={'COUNTRY': True, 'IS_LONG_TERM': True, 'DAYS_ACTIVE': True},
                    color_discrete_map={
                        '⭐ Star Partners': COLORS['accent'],
                        '⚠️ High-Value Switchers': COLORS['warning'],
                        '💚 Loyal Small Buyers': '#17a2b8',
                        '❌ Risky Prospects': COLORS['danger']
                    },
                    title='Importer Stability Quadrant Analysis',
                    labels={'SUPPLIER_LOYALTY_SCORE': 'Supplier Loyalty Score →',
                           'TOTAL_FOB_INR': 'Total Purchase Value (₹) →'},
                    size_max=30)

    # Add quadrant lines
    fig.add_vline(x=median_loyalty, line_dash="dash", line_color="gray")
    fig.add_hline(y=median_fob, line_dash="dash", line_color="gray")

    # Add quadrant annotations
    fig.add_annotation(x=median_loyalty*1.5, y=median_fob*2,
                      text="STAR PARTNERS<br>(Priority Meetings)",
                      showarrow=False, font=dict(size=10, color=COLORS['accent']))

    fig.add_annotation(x=median_loyalty*0.5, y=median_fob*2,
                      text="HIGH-VALUE SWITCHERS<br>(Competitive Pricing)",
                      showarrow=False, font=dict(size=10, color=COLORS['warning']))

    fig.update_layout(height=700, showlegend=True)

    return fig


# Meeting planner
@app.callback(
    Output('meeting-list', 'children'),
    [Input('region-filter', 'value'),
     Input('country-filter', 'value'),
     Input('address-filter', 'value'),
     Input('longterm-filter', 'value'),
     Input('fob-slider', 'value')]
)
def update_meeting_list(region, countries, has_address, is_longterm, min_fob):
    filtered_df = filter_data(region, countries, has_address, is_longterm, min_fob)

    importer_country = gulf_me_transactions.groupby('IMPORTER_NAME')['COUNTRY'].first().reset_index()
    filtered_with_country = filtered_df.merge(importer_country, on='IMPORTER_NAME')
    filtered_with_country = filtered_with_country[filtered_with_country['COUNTRY'].isin(countries)]

    # Calculate priority score
    filtered_with_country['Priority_Score'] = (
        filtered_with_country['TOTAL_FOB_INR'] *
        filtered_with_country['SUPPLIER_LOYALTY_SCORE'] *
        filtered_with_country['IS_LONG_TERM'].map({'Yes': 1.5, 'No': 1.0})
    )

    top_meetings = filtered_with_country.nlargest(30, 'Priority_Score')

    # Group by country for better organization
    cards = []
    for country in top_meetings['COUNTRY'].unique():
        country_importers = top_meetings[top_meetings['COUNTRY'] == country]

        cards.append(
            dbc.Card([
                dbc.CardHeader(html.H5(f"🏙️ {country}", className="mb-0")),
                dbc.CardBody([
                    dbc.ListGroup([
                        dbc.ListGroupItem([
                            html.Div([
                                html.H6(f"{idx}. {row['IMPORTER_NAME']}", className="mb-1"),
                                html.Small([
                                    f"💰 Total FOB: ₹{row['TOTAL_FOB_INR']/1e6:.2f}M | ",
                                    f"📊 Loyalty: {row['SUPPLIER_LOYALTY_SCORE']:.2f} | ",
                                    f"📅 {row['DAYS_ACTIVE']} days active | ",
                                    f"✅ Long-term: {row['IS_LONG_TERM']} | ",
                                    f"📍 Address: {row['HAS_ADDRESS']}"
                                ], className="text-muted"),
                                html.Br(),
                                html.Small(f"Products: {row['HS_CODES_LIST'][:100]}...",
                                         className="text-info")
                            ])
                        ]) for idx, (_, row) in enumerate(country_importers.iterrows(), 1)
                    ], flush=True)
                ])
            ], className="mb-3")
        )

    return cards


# Run the app
if __name__ == '__main__':
    print("\n" + "="*60)
    print("🚀 Starting Gulf & Middle East Prospecting Dashboard")
    print("="*60)
    print("📍 Open your browser and go to: http://127.0.0.1:8050")
    print("💡 Use filters to narrow down your target importers")
    print("📥 Download Excel from 'Top Importers' tab for meeting prep")
    print("="*60 + "\n")

    app.run(debug=True, host='0.0.0.0', port=8050)

📊 Loading data...
✅ Data loaded: 334 Gulf & ME importers, 875 transactions

🚀 Starting Gulf & Middle East Prospecting Dashboard
📍 Open your browser and go to: http://127.0.0.1:8050
💡 Use filters to narrow down your target importers
📥 Download Excel from 'Top Importers' tab for meeting prep

Dash is running on http://0.0.0.0:8050/



INFO:dash.dash:Dash is running on http://0.0.0.0:8050/



 * Serving Flask app '__main__'
 * Debug mode: on


# EXPORT DATA QUALITY CHECKER v2.0

In [ ]:
"""
================================================================================
EXPORT DATA QUALITY CHECKER v2.0
================================================================================
Initial data profiling tool for export data files with varying column names.

THREE WAYS TO LOAD YOUR DATA:
------------------------------

OPTION 1: Upload Files Directly (Recommended for Small Datasets)
   ✅ Best for: 5-20 files, under 100MB total
   ✅ Pros: Simple, no configuration needed
   ✅ How: Just click upload button and select your files

OPTION 2: Google Drive Shared Link (Recommended for Automation)
   ✅ Best for: Automated workflows, shared team folders
   ✅ Pros: No manual upload, works with shared drives
   ✅ Requirements:
      • Folder must be shared (Anyone with link → Viewer)
      • Need the full folder URL
   ✅ How: Paste your Google Drive folder link when prompted

OPTION 3: Mount Google Drive (Recommended for Personal Drive)
   ✅ Best for: Files in your personal Google Drive
   ✅ Pros: Full access to all your Drive folders
   ✅ How: Authorize Drive access, then browse to folder

FEATURES:
---------
• Handles multiple file formats (xlsx, xls, csv)
• Maps columns using semantic synonyms
• Generates quality reports before cleaning
• Saves report in same location as source files

For Google Colab - Choose your preferred method and run!
================================================================================
"""

# ============================================================================
# CELL 1: INSTALL & IMPORT
# ============================================================================

!pip install -q pandas numpy openpyxl xlrd gdown

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import warnings
import os
import re
warnings.filterwarnings('ignore')

print("✅ Dependencies loaded!")
print(f"📅 Session: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ============================================================================
# CELL 2: COLUMN SYNONYM MAPPING
# ============================================================================

# Canonical column names → list of possible variations found in different data sources
COLUMN_SYNONYMS = {
    # Product Information
    "PRODUCT_DESCRIPTION": [
        "PRODUCT DESCRIPTION", "ITEM", "ITEM DESCRIPTION",
        "ITEM DESCRIPTIONS", "PRODUCT_DESCRIPTION", "DESCRIPTION",
        "GOODS DESCRIPTION", "COMMODITY", "PRODUCT", "COMMODITY DESCRIPTION", "PRODUCTDESCRIPITION",
        "RITC DESCRIPTION", "PRODUCTDESCRIPITION","GOODSDESCRIPTION", "HSN_DESCRIPTION", 'ITEM_DESCRIPTION',
        'ITEM_CATEGORY_DESCRIPTION', "ITME", "PRODUCT_DESCRIPITION"
    ],
    "HS_CODE": [
        "HS CODE", "HS_CODE", "HSN_CODE", "RITC", "HSCODE",
        "HSN CODE", "HSN", "TARIFF CODE", "HS", "RITCCODE", "RITC_8", "RITC_CODE"
    ],
    "CHAPTER": [
        "CH", "CHAPTER", "2 DIGIT", "HSCODE(2 DIGIT)", "HS2",
        "CHAPTER CODE", "2DIGIT", "CHAPTER'S","CHEPTER", "CHAPTERS"
    ],
    "HS4": [
        "HS4", "4 DIGIT", "HS 4", "HS_4", "4DIGIT", "HSCODE(4 DIGIT)"
    ],

    # Quantity & Value
    "QUANTITY": [
        "QUANTITY", "PRODUCT_QUANTITY", "QTY", "QTY.", "QNT"
    ],
    "UNIT": [
        "UQC", "UNIT", "QUANTITY_UNIT", "UNIT QUANTITY", "UOM",
        "UNIT OF MEASURE", "QUANTITY UNIT", "UNITOFMEASUREMENT", "UNITQUANTITY","UNIT_QUANTITY", "UNITS"
    ],
    "UNIT_RATE_FC": [
        "ITEM_RATE", "UNIT RATE IN FC", "UNT PRICE FC", "UNIT PRICE (USD)",
        "UNIT RATE", "UNIT PRICE", "RATE", "UNIT_VALUE_USD", "UNIT_VALUE_FC", "UNIT_RATE_USD",
        "UNITPRICE", "UNIT PRICE FOREIGN", "UNIT PRICE FC", "ITEM_RATE_IN_FC","UNIT RATE IN FOREIGN CURRENCY",
        "RATE IN FC", "INV UNIT RATE", "ITEM RATE"
    ],
    "UNIT_RATE_INR": [
        "UNIT RATE IN INR", "UNIT PRICE IN INR", "RATE INR","PER UNIT FOB", "UNT PRICE INR",
        "ITEM_RATE", "UNIT_VALUE_INR", "UNIT_VALUE_IN_INR", "UNIT_RATE_IN_INR", "ITEM RATE IN INR"
    ],
    "FOB_INR": [
        "FOB", "FOB INR", "FOB IN INR", "FOB VALUE (INR)", "FOB_IN_INR",
        "FOB VALUE", "FOB_VALUE", "VALUE INR", "INR VALUE", "FOB IN INR.1", "FOBVALUEINRS",
        "TOTAL_VALUE_IN_INR", "TOTAL FOB VALUE IN INR", "VALUE"
    ],
    "FOB_FC": [
        "FOB IN FC", "INV VALUE FC", "FOB FC", "VALUE IN FC",
        "FOB USD", "USD VALUE", "FC VALUE", "TOTAL VALUE IN FC",
        "TOTAL_VALUE_IN_FC", "TOTAL_VALUE_USD", "TOTAL_VALUE_FC","TOTAL_VALUE_IN_USD", "VALUE IN FC"
    ],
    "CURRENCY": [
        "CURRENCY", "CURR", "CUR", "CURRENCY CODE", "CURR", "CURRENCY_NAME","UNIT RATE CURRENCY", "RATE CURRENCY"
    ],

    # Exporter Information
    "EXPORTER_NAME": [
        "EXPORTER", "EXPORTER NAME", "EXPORTER NAMES", "EXPORTER_NAME",
        "SHIPPER", "SHIPPER NAME", "SUPPLIER", "SELLER", "EXPORTER_PERSON_NAME", "EXPORTERNAME",
        "INDIAN EXPORTER NAME","EXPORTER_NAME"
    ],
    "EXPORTER_ID": [
        "EXPORTER ID", "IEC", "IEC CODE", "IEC NO", "EXPORTER_ID",
        "IE CODE", "IMPORTER EXPORTER CODE", "IECNO", "IEC_NO"
    ],
    "EXPORTER_ADDRESS": [
        "EXPORTER ADDRESS", "Exporter_Address", "EXPORTER ADD",
        "EXPORTER ADD1", "Exporter Add1", "SHIPPER ADDRESS", "SHIPPER'S ADDRESS", "EXPORTER_ADDRESS.1",
        "ADDRESS", "EXPORTER ADDRESS & STATE", "ADRESS", "SHIPPER ADDRESS1"
    ],
    "EXPORTER_CITY_STATE": [
        "EXPORTER CITY/ STATE", "EXPORTER CITY", "Exporter_City_State",
        "EXPORTER STATE", "Exporter City", "EXPORTER CITY/STATE", "CITY STATE", 'EXPORTER_CITY_STATE.1',
        'EXPORTER_STATE','CITY/ STATE',"EXPORTER_CITY_STATE","CITY/ STATE","EXPORTER_CITY_STATE",
        "EXPORTER ADD2", "CITY", "SHIPPER CITY", "SHIPPER STATE"
    ],
    "EXPORTER_PINCODE": [
        "EXPORTER PIN", "Exporter_PIN", "EXPORTER PINCODE", "PIN CODE", "PIN", "EXPORTER_PIN",
        "PIN_CODE", 'EXPORTER PIN CODE','EXPORTER_PINCODE'
    ],
    "EXPORTER_CONTACT_PERSON": [
        "CONTACT PERSON", "CONTACT PERSON2", "Exporter_Person_Name", "CONTACT PERSON NAME",
        "SHIPPER CONTACT PERSON"
    ],
    "EXPORTER_CONTACT_EMAIL": [
        "EMAIL ID", "Exporter_Email", "EMAIL", "E-MAIL", "EXPORTER EMAIL", "EXPORTER_EMAIL", "EMAILID",
        "EMAIL", "SHIPPER EMAIL"
    ],
    "EXPORTER_CONTACT_PHONE": [
        "CONTACT NO.", "Exporter_Contact", "PHONE", "MOBILE", "CONTACT", "EXPORTER PHONE", "EXPORTER_PHONE",
        "PHONE", "EXPORTER_CONTACT", 'CONTACTNO', "SHIPPER PHONE"
    ],

    # Importer Information
    "IMPORTER_NAME": [
        "IMPORTER NAME", "IMPORTER NAMES", "IMPORTER_NAME", "CONSIGNEE",
        "IMPORTER", "BUYER", "BUYER NAME", "IMPORTER NAME ", "CONSIGNEE NAME",'CONSINEENAME','CONSIGNEENAME',
        "CONSINEE_NAME", "CONSIGNEE_NAME","FOREIGN IMPORTER NAME","CONSINEE_NAME","FOREIGN IMPORTER NAME ADDRESS"
    ],
    "IMPORTER_ADDRESS": [
        "IMPORTER ADDRESS", "Consignee_Address", "BUYER ADDRESS", "CONSIGNEE_ADDRESS",
        "CONSINEEADDRESS","CONSIGNEE_ADDRESS4", "ADDRESS2", "CONSIGNEE ADD","CONSIGNEEADDRESS",
        "CONSINEE_ADDRESS","FOR_ADD1", "CONSIGNEE ADD1", "CONSIGNEE ADD2"
    ],
    "IMPORTER_CITY": [
        "IMPORTER CITY", "CONSIGNEE CITY", "BUYER CITY", "CONSIGNEE_CITY"
    ],

    # Port & Location
    "INDIAN_PORT": [
        "PORT CODE", "ORIGIN PORT", "INDIAN PORT",
        "ORIGIN_PORT_CODE", "CUSH", "PORT", "LOADING PORT", "PORT OF LOADING",
        "LOCATION1", "LOCATION", "SOURCE_PORT", 'INDIAN_PORT', "INDIAN PORT NAME","PORT OF ORIGIN"
    ],
    "FOREIGN_PORT": [
        "FOREIGN PORT", "DESTINATION PORT", "DISCHARGE PORT",
        "POD", "PORT_CD", "PORT OF DISCHARGE",
        "DESTINATION_PORT", "FORIGN PORT",
        "FOREIGNPORT", "PORT OF DESTINATION","FOREIGN_PORT", "PORT CD"
    ],
    "COUNTRY": [
        "COUNTRY", "FOREIGN COUNTRY", "DESTINATION_COUNTRY",
        "DEST COUNTRY", "DESTINATION", "COUNTRY OF DESTINATION",
        "COUNTRY OF ORIGIN", "CONSIGNEE COUNTRY", "SOURCE_COUNTRY", "ORIGIN_COUNTRY",
        "FOREIGNCOUNTRY", "COUNTRYOFDESTINATIONNAME", "FOREIGN_COUNTRY", "CTRY OF DESTINATION"
    ],
    "MODE_OF_PORT": [
        "MODE OF PORT", "SHIPMENT MODE", "MODE", "TRANSPORT MODE", "MODE_OF_TRANSPORT"
    ],

    # Date & Time
    "SB_DATE": [
        "SBDT", "SB DATE", "SBDATE", "SHIPPING_DATE", "SHIPPING BILL DATE",
        "SHIPPING DATE", "DATE", "EXPORT DATE", "SB_DATE", "DATE", "SB_DT"
    ],
    "MONTH": ["MONTH", "MON", "MM"],
    "YEAR": ["YEAR", "YR", "YYYY"],

    # Document References
    "SB_NO": [
        "SBNO", "SB NO", "SHIPPING BILL NO", "SHIPPING_BILL_NO",
        "SB NUMBER", "BILL NO", "SBNUMBER", "SYSTEM_ID", "SB.NO."
    ],
    "INVOICE_NO": [
        "INVOICE_NO", "INVOICE NO", "INV NO", "INVOICE NUMBER", "INVOICE NO.",'INVOICE_NUMBER'
    ],
    "ITEM_NO": ["ITEM_NO", "ITEM NO", "LINE NO", "SR NO", "ITEM NUMBER"],

    # Other
    "DRAWBACK": ["DRAWBACK", "DWARBACK", "DBK", "DRAWBAKDVALUE", "DWARBACK"],
    "TYPE": ["TYPE"],
    "UID": ["UID"],
    "ID": ["ID"],
}

print(f"✅ Column mappings loaded: {len(COLUMN_SYNONYMS)} canonical columns")

# ============================================================================
# CELL 3: CHOOSE YOUR DATA SOURCE
# ============================================================================

print("="*60)
print("📂 DATA SOURCE OPTIONS")
print("="*60)
print("\nChoose how you want to load your data:")
print("1. Upload files directly (recommended for small datasets)")
print("2. Use Google Drive shared link with gdown (for automation)")
print("3. Mount Google Drive and browse folders")
print("="*60)

choice = input("\nEnter your choice (1, 2, or 3): ").strip()

# ============================================================================
# CELL 4: LOAD DATA BASED ON CHOICE
# ============================================================================

if choice == "1":
    # OPTION 1: Upload files directly
    print("\n" + "="*60)
    print("📤 UPLOAD YOUR DATA FILES")
    print("="*60)

    from google.colab import files
    print("Please upload your export data files (xlsx, xls, csv)")
    print("You can select multiple files at once")
    print("="*60)

    uploaded = files.upload()

    # Save to /content/data_files/
    FOLDER_PATH = '/content/data_files'
    os.makedirs(FOLDER_PATH, exist_ok=True)

    for filename, content in uploaded.items():
        filepath = os.path.join(FOLDER_PATH, filename)
        with open(filepath, 'wb') as f:
            f.write(content)
        print(f"✅ Saved: {filename}")

    print(f"\n📁 Total files uploaded: {len(uploaded)}")
    print(f"📍 Files location: {FOLDER_PATH}")

elif choice == "2":
    # OPTION 2: Download from Google Drive shared link using gdown
    print("\n" + "="*60)
    print("🔗 DOWNLOAD FROM GOOGLE DRIVE SHARED LINK")
    print("="*60)

    !pip install -q gdown
    import gdown

    print("\n📌 IMPORTANT: Your Google Drive link must be:")
    print("   • Publicly accessible (Anyone with link can VIEW)")
    print("   • A folder link (not individual file)")
    print("")
    print("Example: https://drive.google.com/drive/folders/1xfn0Sq...")
    print("="*60)

    drive_url = input("\nPaste your Google Drive folder URL:\n> ").strip()

    FOLDER_PATH = '/content/data_files'

    print(f"\n⏳ Downloading files from Google Drive...")
    try:
        gdown.download_folder(
            url=drive_url,
            output=FOLDER_PATH,
            quiet=False,
            use_cookies=False
        )
        print(f"✅ Download complete!")
        print(f"📍 Files saved to: {FOLDER_PATH}")

        # List downloaded files
        files_found = [f for f in os.listdir(FOLDER_PATH)
                       if f.endswith(('.xlsx', '.xls', '.csv'))]
        print(f"📊 Found {len(files_found)} data files")

    except Exception as e:
        print(f"❌ Download failed: {str(e)}")
        print("\n💡 TROUBLESHOOTING:")
        print("   1. Make sure link sharing is enabled (Anyone with link → Viewer)")
        print("   2. Use the folder link, not file link")
        print("   3. Try Option 1 (Upload files) instead")

elif choice == "3":
    # OPTION 3: Mount Google Drive and browse
    print("\n" + "="*60)
    print("📂 MOUNT GOOGLE DRIVE")
    print("="*60)

    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted!")

    print("\n" + "="*60)
    print("📁 SPECIFY DATA FOLDER PATH")
    print("="*60)
    print("\nExample: /content/drive/MyDrive/Export_Data/2024")
    print("💡 TIP: Right-click folder in Drive sidebar → Copy path")
    print("="*60)

    FOLDER_PATH = input("\nEnter the full path to your data folder:\n> ").strip()

    # Validate path
    if not os.path.exists(FOLDER_PATH):
        print(f"❌ Folder not found: {FOLDER_PATH}")
        print("\n💡 Make sure you copied the full path correctly")
    else:
        print(f"✅ Folder found: {FOLDER_PATH}")

        # List files
        files_found = [f for f in os.listdir(FOLDER_PATH)
                       if f.endswith(('.xlsx', '.xls', '.csv'))]
        print(f"📊 Found {len(files_found)} data files")

        if files_found:
            print("\nFiles preview:")
            for i, f in enumerate(files_found[:10], 1):
                print(f"   {i}. {f}")
            if len(files_found) > 10:
                print(f"   ... and {len(files_found) - 10} more")

else:
    print("❌ Invalid choice. Please run again and enter 1, 2, or 3")
    FOLDER_PATH = None

# ============================================================================
# CELL 5: CORE FUNCTIONS
# ============================================================================

def get_file_size_mb(filepath):
    """Get file size in MB."""
    return os.path.getsize(filepath) / (1024 * 1024)


def extract_date_range(df):
    """Extract date range from any date column in the dataframe."""
    date_keywords = ["DATE", "DT", "SBDT"]
    date_cols = [c for c in df.columns if any(kw in c.upper() for kw in date_keywords)]

    min_date, max_date = None, None

    for col in date_cols:
        parsed = pd.to_datetime(df[col], errors='coerce')
        valid_dates = parsed.dropna()

        if len(valid_dates) > 0:
            col_min = valid_dates.min()
            col_max = valid_dates.max()

            if min_date is None or col_min < min_date:
                min_date = col_min
            if max_date is None or col_max > max_date:
                max_date = col_max

    return min_date, max_date


def extract_months_years(df):
    """Extract unique months and years from date columns."""
    date_keywords = ["DATE", "DT", "SBDT"]
    date_cols = [c for c in df.columns if any(kw in c.upper() for kw in date_keywords)]

    months = set()
    years = set()

    for col in date_cols:
        parsed = pd.to_datetime(df[col], errors='coerce')
        valid_dates = parsed.dropna()

        if len(valid_dates) > 0:
            months.update(valid_dates.dt.to_period('M').astype(str).unique())
            years.update(valid_dates.dt.year.unique())

    return sorted(months), sorted(years)


def map_columns_to_canonical(df_columns, synonym_map):
    """
    Map actual columns to canonical names.
    Returns: dict of {canonical_name: actual_column_name} and list of unmapped columns
    """
    df_cols_upper = {col: col for col in df_columns}
    mapped = {}
    used_columns = set()

    # Map to canonical columns
    for canonical, variants in synonym_map.items():
        variants_upper = [v.upper() for v in variants]

        for actual_col in df_columns:
            if actual_col.upper() in variants_upper:
                mapped[canonical] = actual_col
                used_columns.add(actual_col)
                break  # Take first match only

    # Find unmapped columns
    unmapped = [col for col in df_columns if col not in used_columns]

    return mapped, unmapped


def read_all_files(folder_path):
    """Read all Excel/CSV files from a folder into a dictionary of DataFrames."""
    dataframes = {}

    for file in Path(folder_path).iterdir():
        try:
            if file.suffix.lower() in [".xlsx", ".xls"]:
                df = pd.read_excel(file)
            elif file.suffix.lower() == ".csv":
                # Try different encodings
                for encoding in ['utf-8', 'latin-1', 'cp1252']:
                    try:
                        df = pd.read_csv(file, encoding=encoding)
                        break
                    except:
                        continue
            else:
                continue

            # Standardize column names
            df.columns = (
                df.columns
                .astype(str)
                .str.strip()
                .str.upper()
            )

            dataframes[file.name] = {
                'df': df,
                'path': str(file)
            }
            print(f"  ✅ {file.name}: {len(df):,} rows × {df.shape[1]} columns")

        except Exception as e:
            print(f"  ❌ {file.name}: Error - {str(e)[:50]}")

    return dataframes


def create_master_summary(dfs_dict, synonym_map):
    """
    Create the master summary sheet with file metadata and column mappings.
    Format: FILE_NAME | SIZE_MB | ROWS | COLUMNS | NULL_COUNT | DATE_RANGE |
            MONTHS | YEARS | CANONICAL_COL_1 | ... | CANONICAL_COL_N | UNDEFINED_1 | ...
    """
    rows = []

    # Get all canonical columns in order
    canonical_cols = sorted(synonym_map.keys())

    # Track max undefined columns needed
    max_undefined = 0

    for filename, data in dfs_dict.items():
        df = data['df']
        filepath = data['path']

        # Basic metadata
        file_size_mb = get_file_size_mb(filepath)
        total_rows = len(df)
        total_cols = df.shape[1]
        null_count = df.isnull().sum().sum()

        # Date range
        min_date, max_date = extract_date_range(df)
        date_range = f"{min_date.strftime('%Y-%m-%d') if min_date else 'N/A'} to {max_date.strftime('%Y-%m-%d') if max_date else 'N/A'}"

        # Months and years
        months, years = extract_months_years(df)
        months_str = ', '.join(months) if months else 'N/A'
        years_str = ', '.join(map(str, years)) if years else 'N/A'

        # Column mapping
        mapped, unmapped = map_columns_to_canonical(df.columns, synonym_map)

        # Track max undefined columns
        if len(unmapped) > max_undefined:
            max_undefined = len(unmapped)

        # Build row
        row = {
            'FILE_NAME': filename,
            'SIZE_MB': round(file_size_mb, 2),
            'ROWS': total_rows,
            'COLUMNS': total_cols,
            'NULL_COUNT': null_count,
            'DATE_RANGE': date_range,
            'MONTHS': months_str,
            'YEARS': years_str,
        }

        # Add canonical columns
        for canonical in canonical_cols:
            row[canonical] = mapped.get(canonical, '')

        # Add undefined columns
        for i, unmapped_col in enumerate(unmapped, 1):
            row[f'UNDEFINED_{i}'] = unmapped_col

        rows.append(row)

    # Create DataFrame
    summary_df = pd.DataFrame(rows)

    # Ensure all UNDEFINED columns exist (fill with empty for files with fewer undefined cols)
    for i in range(1, max_undefined + 1):
        col_name = f'UNDEFINED_{i}'
        if col_name not in summary_df.columns:
            summary_df[col_name] = ''

    # Reorder columns: metadata first, then canonical, then undefined
    metadata_cols = ['FILE_NAME', 'SIZE_MB', 'ROWS', 'COLUMNS', 'NULL_COUNT',
                     'DATE_RANGE', 'MONTHS', 'YEARS']
    undefined_cols = [c for c in summary_df.columns if c.startswith('UNDEFINED_')]
    undefined_cols = sorted(undefined_cols, key=lambda x: int(x.split('_')[1]))

    final_cols = metadata_cols + canonical_cols + undefined_cols
    summary_df = summary_df[final_cols]

    return summary_df


def value_field_analysis(df):
    """Analyze numeric/value fields for zeros, negatives, and outliers."""
    results = {}

    value_keywords = ["QUANTITY", "FOB", "RATE", "PRICE", "VALUE", "QTY", "AMOUNT"]
    value_cols = [
        c for c in df.columns
        if any(kw in c.upper() for kw in value_keywords)
    ]

    for col in value_cols:
        try:
            # Convert to numeric if not already
            if not pd.api.types.is_numeric_dtype(df[col]):
                numeric_col = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')
            else:
                numeric_col = df[col]

            total = len(numeric_col)
            non_null = numeric_col.notna().sum()

            if non_null > 0:
                results[col] = {
                    'TOTAL': total,
                    'NON_NULL': non_null,
                    'NULL': total - non_null,
                    'ZEROS': (numeric_col == 0).sum(),
                    'ZERO_%': round((numeric_col == 0).sum() / total * 100, 2),
                    'NEGATIVES': (numeric_col < 0).sum(),
                    'MIN': float(numeric_col.min()) if pd.notna(numeric_col.min()) else 0,
                    'MAX': float(numeric_col.max()) if pd.notna(numeric_col.max()) else 0,
                    'MEAN': round(float(numeric_col.mean()), 2) if pd.notna(numeric_col.mean()) else 0,
                    'MEDIAN': round(float(numeric_col.median()), 2) if pd.notna(numeric_col.median()) else 0,
                }
        except Exception as e:
            print(f"  ⚠️ Warning: Could not analyze value field {col}: {str(e)[:50]}")
            continue

    return results


def date_field_analysis(df):
    """Analyze date fields for validity and range."""
    results = {}

    date_keywords = ["DATE", "DT", "SBDT"]
    date_cols = [c for c in df.columns if any(kw in c.upper() for kw in date_keywords)]

    for col in date_cols:
        try:
            parsed = pd.to_datetime(df[col], errors='coerce')
            valid = parsed.notna()

            results[col] = {
                'TOTAL': len(df),
                'VALID_DATES': int(valid.sum()),
                'INVALID_DATES': int((~valid).sum()),
                'VALID_%': round(valid.sum() / len(df) * 100, 2),
                'MIN_DATE': parsed.min().strftime('%Y-%m-%d') if valid.any() and pd.notna(parsed.min()) else 'N/A',
                'MAX_DATE': parsed.max().strftime('%Y-%m-%d') if valid.any() and pd.notna(parsed.max()) else 'N/A',
            }
        except Exception as e:
            print(f"  ⚠️ Warning: Could not analyze date field {col}: {str(e)[:50]}")
            continue

    return results


def text_field_analysis(df, synonym_map):
    """Analyze key text fields for uniqueness and patterns."""
    results = {}

    key_fields = ['EXPORTER_NAME', 'IMPORTER_NAME', 'COUNTRY', 'HS_CODE', 'INDIAN_PORT', 'EXPORTER_ID']

    for field_name in key_fields:
        if field_name not in synonym_map:
            continue

        possible_cols = [v.upper() for v in synonym_map[field_name]]
        matching_cols = [col for col in df.columns if col in possible_cols]

        if matching_cols:
            col = matching_cols[0]

            # Convert to string and clean - handle Series correctly
            try:
                # Get the column as a Series
                col_data = df[col]

                # Convert to string, then apply str operations
                series = col_data.astype(str).str.strip().str.upper()

                results[field_name] = {
                    'COLUMN_FOUND': col,
                    'ALL_MATCHES': matching_cols,
                    'TOTAL': len(series),
                    'UNIQUE': series.nunique(),
                    'NULL/EMPTY': series.isin(['', 'NAN', 'NONE', 'NA', '-']).sum(),
                    'TOP_5': series.value_counts().head(5).to_dict()
                }
            except Exception as e:
                # If there's an error, skip this field
                print(f"  ⚠️ Warning: Could not analyze {field_name}: {str(e)[:50]}")
                continue

    return results

print("✅ Analysis functions loaded!")

# ============================================================================
# CELL 6: RUN ANALYSIS
# ============================================================================

# Validate that we have a folder path
if FOLDER_PATH is None or not os.path.exists(FOLDER_PATH):
    print("\n❌ ERROR: No valid data folder found!")
    print("Please run the previous cell again and choose a valid option.")
else:
    print("\n" + "="*60)
    print("📊 RUNNING DATA QUALITY ANALYSIS")
    print("="*60)

    # Read all files
    print("\n📂 Loading files...")
    dfs = read_all_files(FOLDER_PATH)

if not dfs:
    print("❌ No valid files found!")
else:
    print(f"\n✅ Loaded {len(dfs)} file(s)")

    # Create master summary
    print("\n" + "-"*60)
    print("📋 CREATING MASTER SUMMARY")
    print("-"*60)
    master_summary = create_master_summary(dfs, COLUMN_SYNONYMS)
    print(f"✅ Master summary created: {len(master_summary)} files × {len(master_summary.columns)} columns")

    # Show preview
    print("\nPreview (first 3 rows, first 15 columns):")
    print(master_summary.iloc[:3, :15].to_string(index=False))

    # Generate additional analytics
    print("\n" + "-"*60)
    print("📊 GENERATING ADDITIONAL ANALYTICS")
    print("-"*60)

    value_rows = []
    date_rows = []
    text_rows = []

    for filename, data in dfs.items():
        df = data['df']

        # Value analysis
        value_analysis = value_field_analysis(df)
        for col, vals in value_analysis.items():
            row = {'FILE': filename, 'COLUMN': col, **vals}
            value_rows.append(row)

        # Date analysis
        date_analysis = date_field_analysis(df)
        for col, vals in date_analysis.items():
            row = {'FILE': filename, 'COLUMN': col, **vals}
            date_rows.append(row)

        # Text analysis
        text_analysis = text_field_analysis(df, COLUMN_SYNONYMS)
        for field, vals in text_analysis.items():
            row = {'FILE': filename, 'FIELD': field, **vals}
            text_rows.append(row)

    value_df = pd.DataFrame(value_rows) if value_rows else pd.DataFrame()
    date_df = pd.DataFrame(date_rows) if date_rows else pd.DataFrame()
    text_df = pd.DataFrame(text_rows) if text_rows else pd.DataFrame()

    print(f"✅ Value analysis: {len(value_df)} records")
    print(f"✅ Date analysis: {len(date_df)} records")
    print(f"✅ Text analysis: {len(text_df)} records")

# ============================================================================
# CELL 7: SAVE REPORT
# ============================================================================

print("\n" + "="*60)
print("💾 SAVING REPORT")
print("="*60)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_filename = f'Data_Quality_Report_{timestamp}.xlsx'

# Save in the same folder as the data files
output_path = os.path.join(FOLDER_PATH, output_filename)

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # Sheet 1: Master Summary (custom format)
    master_summary.to_excel(writer, sheet_name='Master_Summary', index=False)

    # Sheet 2: Value Analysis
    if not value_df.empty:
        value_df.to_excel(writer, sheet_name='Value_Analysis', index=False)

    # Sheet 3: Date Analysis
    if not date_df.empty:
        date_df.to_excel(writer, sheet_name='Date_Analysis', index=False)

    # Sheet 4: Text Field Analysis
    if not text_df.empty:
        text_df.to_excel(writer, sheet_name='Text_Analysis', index=False)

    # Sheet 5: Column Presence Matrix
    all_columns = sorted(set(col for data in dfs.values() for col in data['df'].columns))
    presence = pd.DataFrame(index=all_columns)

    for filename, data in dfs.items():
        short_name = filename[:30] + "..." if len(filename) > 30 else filename
        presence[short_name] = presence.index.isin(data['df'].columns)

    presence['FILES_WITH_COLUMN'] = presence.sum(axis=1)
    presence = presence.sort_values('FILES_WITH_COLUMN', ascending=False)
    presence.to_excel(writer, sheet_name='Column_Presence')

print(f"✅ Report saved: {output_path}")

# Also download to local machine
from google.colab import files
try:
    files.download(output_path)
    print(f"✅ Report downloaded to your computer")
except:
    print(f"⚠️ Could not auto-download. File saved at: {output_path}")

# ============================================================================
# CELL 8: SUMMARY
# ============================================================================

print("\n" + "="*60)
print("🎉 DATA QUALITY CHECK COMPLETE!")
print("="*60)

# Count canonical columns found
canonical_found = sum(
    1 for col in COLUMN_SYNONYMS.keys()
    if col in master_summary.columns and master_summary[col].notna().any()
)

# Count undefined columns
undefined_cols = [c for c in master_summary.columns if c.startswith('UNDEFINED_')]

print(f"""
📊 SUMMARY
==========
Files Analyzed: {len(dfs)}
Total Records: {sum(len(data['df']) for data in dfs.values()):,}

Column Mapping:
  • Canonical columns found: {canonical_found}/{len(COLUMN_SYNONYMS)}
  • Unmapped columns: {len(undefined_cols)} column slots created

File Location: {FOLDER_PATH}
Report Location: {output_path}

📁 REPORT STRUCTURE:
   • Master_Summary - File metadata + all column mappings (canonical & undefined)
   • Value_Analysis - Numeric field quality checks
   • Date_Analysis - Date field validity and ranges
   • Text_Analysis - Key text field uniqueness
   • Column_Presence - Which columns exist in which files

💡 NEXT STEPS:
1. Review Master_Summary sheet for complete file overview
2. Check unmapped columns (UNDEFINED_*) to identify missing synonyms
3. Review Value_Analysis for data quality issues
4. Use column mappings to standardize schema across files
""")

✅ Dependencies loaded!
📅 Session: 2026-02-11 04:48:43
✅ Column mappings loaded: 36 canonical columns
📂 DATA SOURCE OPTIONS

Choose how you want to load your data:
1. Upload files directly (recommended for small datasets)
2. Use Google Drive shared link with gdown (for automation)
3. Mount Google Drive and browse folders

Enter your choice (1, 2, or 3): 2

🔗 DOWNLOAD FROM GOOGLE DRIVE SHARED LINK

📌 IMPORTANT: Your Google Drive link must be:
   • Publicly accessible (Anyone with link can VIEW)
   • A folder link (not individual file)

Example: https://drive.google.com/drive/folders/1xfn0Sq...

Paste your Google Drive folder URL:
> https://drive.google.com/drive/folders/1SlqUml1K4i7opC4PHk5KDS5qBZET0A6B?usp=drive_link

⏳ Downloading files from Google Drive...


Retrieving folder contents


Processing file 1EX43g8pg5X5s31FfsSUwjIcoi1yAEOOt CH 1 TO 40 EXPORT DEC 22.xlsx
Processing file 1zmGVZx6mOnYU_zEtNugkT5dt_1L-6c5m CH 41 TO 70 EXPORT DEC 22.xlsx
Processing file 1tC_6ZQgPFPSuBTC6MMWsM-OR5l8gldUN CH 71 TO 83 EXPORT DEC 22.xlsx
Processing file 1lixKTlz_rMdbNXsz-v2BeI0Tv8_MxQOf CH 84 EXPORT DEC 22.xlsx
Processing file 13bxrl7vf_yY7UqMYg-9AZgTCW2G9if4I CH 85 EXPORT DEC 22.xlsx
Processing file 1s0UC1B3nJANsIQnkAUxkPgjaRUsgQrKp CH 86 TO 98 EXPORT DEC 22.xlsx


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1EX43g8pg5X5s31FfsSUwjIcoi1yAEOOt
From (redirected): https://drive.google.com/uc?id=1EX43g8pg5X5s31FfsSUwjIcoi1yAEOOt&confirm=t&uuid=dcd6f704-5056-446b-95a9-528bf18b2886
To: /content/data_files/CH 1 TO 40 EXPORT DEC 22.xlsx
100%|██████████| 109M/109M [00:02<00:00, 51.7MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1zmGVZx6mOnYU_zEtNugkT5dt_1L-6c5m
From (redirected): https://drive.google.com/uc?id=1zmGVZx6mOnYU_zEtNugkT5dt_1L-6c5m&confirm=t&uuid=8b14fa2e-319b-41b0-afb1-5a64ccd18ad5
To: /content/data_files/CH 41 TO 70 EXPORT DEC 22.xlsx
100%|██████████| 116M/116M [00:02<00:00, 48.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1tC_6ZQgPFPSuBTC6MMWsM-OR5l8gldUN
From (redirected): https://drive.google.com/uc?id=1tC_6ZQgPFPSuBTC6MMWsM-OR5l8gldUN&confirm=t&uuid=e67c09fe-ea7f-4d55-976

✅ Download complete!
📍 Files saved to: /content/data_files
📊 Found 6 data files
✅ Analysis functions loaded!

📊 RUNNING DATA QUALITY ANALYSIS

📂 Loading files...
  ✅ CH 84 EXPORT DEC 22.xlsx: 493,614 rows × 26 columns
  ✅ CH 71 TO 83 EXPORT DEC 22.xlsx: 977,648 rows × 24 columns
  ✅ CH 41 TO 70 EXPORT DEC 22.xlsx: 977,408 rows × 24 columns
  ✅ CH 1 TO 40 EXPORT DEC 22.xlsx: 893,682 rows × 24 columns
  ✅ CH 85 EXPORT DEC 22.xlsx: 314,137 rows × 26 columns
  ✅ CH 86 TO 98 EXPORT DEC 22.xlsx: 852,771 rows × 24 columns

✅ Loaded 6 file(s)

------------------------------------------------------------
📋 CREATING MASTER SUMMARY
------------------------------------------------------------
✅ Master summary created: 6 files × 46 columns

Preview (first 3 rows, first 15 columns):
                     FILE_NAME  SIZE_MB   ROWS  COLUMNS  NULL_COUNT               DATE_RANGE                                                                                                     MONTHS YEARS CHAPTER COUNTR

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Report downloaded to your computer

🎉 DATA QUALITY CHECK COMPLETE!

📊 SUMMARY
Files Analyzed: 6
Total Records: 4,509,260

Column Mapping:
  • Canonical columns found: 36/36
  • Unmapped columns: 2 column slots created

File Location: /content/data_files
Report Location: /content/data_files/Data_Quality_Report_20260211_051541.xlsx

📁 REPORT STRUCTURE:
   • Master_Summary - File metadata + all column mappings (canonical & undefined)
   • Value_Analysis - Numeric field quality checks
   • Date_Analysis - Date field validity and ranges
   • Text_Analysis - Key text field uniqueness
   • Column_Presence - Which columns exist in which files

💡 NEXT STEPS:
1. Review Master_Summary sheet for complete file overview
2. Check unmapped columns (UNDEFINED_*) to identify missing synonyms
3. Review Value_Analysis for data quality issues
4. Use column mappings to standardize schema across files

